# Пять вариантов ансамбля ДЗ2 NLP

Один private Run All: каждый вариант — cold test1919, затем cold tune2048 на одной видимой T4. Budget640, generation guard160с; timer включает весь pipeline. V4 — primary E2. Никакого обучения или отправки в бота.

In [ ]:
import os, sys, json, hashlib
from pathlib import Path
DRY = os.environ.get("HW2_VARIANTS_DRY_RUN") == "1"
ROOT = Path(os.environ.get("HW2_VARIANTS_ROOT", "/kaggle/working"))
ROOT.mkdir(parents=True, exist_ok=True)
CODE = ROOT / "variants-src" / "hw2_codegen"
CODE.mkdir(parents=True, exist_ok=True)
INPUTS = ROOT / "variants-inputs"
INPUTS.mkdir(parents=True, exist_ok=True)


In [ ]:
import base64, zlib
CODE_FILES = {'__init__.py': '"""Чистая сдача ДЗ2 NLP. Точка входа: hw2_codegen.pipeline.run_inference."""\n', '_numeric.py': '"""AST policy и числовой solve в ограниченном процессе; модуль не импортирует Torch."""\n\nimport ast\nimport builtins\nimport math\nimport random\nimport resource\nimport signal\nimport sys\nfrom types import SimpleNamespace\n\nSAFE_BUILTINS = {\n    n: getattr(builtins, n)\n    for n in (\n        "abs",\n        "all",\n        "any",\n        "bool",\n        "dict",\n        "divmod",\n        "enumerate",\n        "float",\n        "int",\n        "len",\n        "list",\n        "max",\n        "min",\n        "pow",\n        "range",\n        "reversed",\n        "round",\n        "set",\n        "sorted",\n        "str",\n        "sum",\n        "tuple",\n        "zip",\n        "Exception",\n        "ValueError",\n        "TypeError",\n        "ZeroDivisionError",\n        "OverflowError",\n        "ArithmeticError",\n    )\n}\nFORBIDDEN_NAMES = {\n    "open",\n    "input",\n    "eval",\n    "exec",\n    "compile",\n    "globals",\n    "locals",\n    "vars",\n    "getattr",\n    "setattr",\n    "delattr",\n    "dir",\n    "help",\n    "breakpoint",\n    "exit",\n    "quit",\n    "os",\n    "sys",\n    "subprocess",\n    "socket",\n    "ctypes",\n    "importlib",\n    "builtins",\n    "resource",\n    "signal",\n    "pickle",\n    "marshal",\n    "memoryview",\n    "type",\n    "object",\n    "super",\n}\nFORBIDDEN_ATTRS = {\n    "mro",\n    "subclasses",\n    "format_map",\n    "gi_frame",\n    "f_globals",\n    "f_locals",\n    "tb_frame",\n    "func_globals",\n    "code",\n    "modules",\n    "system",\n    "popen",\n    "load",\n    "loads",\n    "dump",\n    "dumps",\n    "random",\n    "format",\n}\nSAFE_BUILTINS["print"] = lambda *args, **kwargs: None\nALLOWED_IMPORTS = {"math", "sympy"}\nSYM_NAMES = {\n    "Symbol",\n    "symbols",\n    "Rational",\n    "Integer",\n    "Float",\n    "Eq",\n    "solve",\n    "sqrt",\n    "simplify",\n}\nMAX_CODE_BYTES = 100000\nMAX_AST_NODES = 5000\n\n\nclass ProgramTimeout(Exception):\n    pass\n\n\ndef filter_program(code):\n    """Проверить AST до исполнения: imports, имена, атрибуты, размер."""\n    if not isinstance(code, str) or len(code.encode("utf-8")) > MAX_CODE_BYTES:\n        return (None, "blocked_size")\n    try:\n        tree = ast.parse(code)\n    except (SyntaxError, ValueError, RecursionError):\n        return (None, "syntax_error")\n    nodes = list(ast.walk(tree))\n    if len(nodes) > MAX_AST_NODES:\n        return (None, "blocked_size")\n    for n in nodes:\n        if isinstance(n, ast.Attribute) and isinstance(n.ctx, (ast.Store, ast.Del)):\n            return (None, "blocked_attribute_mutation")\n        if isinstance(n, ast.Name) and (\n            n.id.startswith("__") or n.id in FORBIDDEN_NAMES\n        ):\n            return (None, "blocked_name")\n        if isinstance(n, ast.Attribute) and (\n            n.attr.startswith("_") or n.attr in FORBIDDEN_ATTRS\n        ):\n            return (None, "blocked_attribute")\n        if isinstance(\n            n, (ast.ClassDef, ast.AsyncFunctionDef, ast.Await, ast.Yield, ast.YieldFrom)\n        ):\n            return (None, "blocked_construct")\n        if isinstance(n, (ast.FunctionDef, ast.arg)):\n            name = n.name if isinstance(n, ast.FunctionDef) else n.arg\n            if name.startswith("__") or name in FORBIDDEN_NAMES:\n                return (None, "blocked_name")\n        if isinstance(n, ast.Import):\n            if any(\n                a.name not in ALLOWED_IMPORTS or (a.asname and a.asname.startswith("_"))\n                for a in n.names\n            ):\n                return (None, "blocked_import")\n        if isinstance(n, ast.ImportFrom):\n            if n.level or n.module not in ALLOWED_IMPORTS:\n                return (None, "blocked_import")\n            if any(\n                a.name.startswith("_")\n                or a.name == "*"\n                or (a.asname and a.asname.startswith("_"))\n                for a in n.names\n            ):\n                return (None, "blocked_import")\n            if n.module == "sympy" and any(a.name not in SYM_NAMES for a in n.names):\n                return (None, "blocked_import")\n    return (tree, None)\n\n\ndef safe_sympy():\n    import sympy\n    from sympy.parsing import sympy_parser\n\n    def denied_parser(*args, **kwargs):\n        raise ValueError("String expression parsing forbidden")\n\n    sympy_parser.parse_expr = denied_parser\n    from sympy.core.cache import clear_cache\n\n    clear_cache()\n    from sympy.core.random import seed\n\n    seed(0)\n\n    def no_strings(x):\n        if isinstance(x, str):\n            raise ValueError("String expressions forbidden")\n        if isinstance(x, (tuple, list)):\n            for y in x:\n                no_strings(y)\n        if isinstance(x, dict):\n            for k, v in x.items():\n                no_strings(k)\n                no_strings(v)\n\n    def wrapped(function):\n\n        def call(*args, **kwargs):\n            no_strings(args)\n            no_strings(list(kwargs.values()))\n            return function(*args, **kwargs)\n\n        return call\n\n    def symbol(name, *args, **kwargs):\n        if (\n            not isinstance(name, str)\n            or not name\n            or (not name.replace("_", "").isalnum())\n        ):\n            raise ValueError("Bad symbol")\n        return sympy.Symbol(name, *args, **kwargs)\n\n    def symbols(names, *args, **kwargs):\n        if not isinstance(names, str) or any(\n            not (c.isalnum() or c in " _,:") for c in names\n        ):\n            raise ValueError("Bad symbols")\n        return sympy.symbols(names, *args, **kwargs)\n\n    return SimpleNamespace(\n        Symbol=symbol,\n        symbols=symbols,\n        **{n: wrapped(getattr(sympy, n)) for n in SYM_NAMES - {"Symbol", "symbols"}},\n    )\n\n\n_MODULES = {}\n\n\ndef safe_import(name, globals=None, locals=None, fromlist=(), level=0):\n    if level or name not in ALLOWED_IMPORTS:\n        raise ImportError("Blocked import")\n    if name not in _MODULES:\n        _MODULES[name] = (\n            SimpleNamespace(\n                **{n: getattr(math, n) for n in dir(math) if not n.startswith("_")}\n            )\n            if name == "math"\n            else safe_sympy()\n        )\n    return _MODULES[name]\n\n\ndef worker_init(memory_bytes):\n    """Лимит address space на Linux и signal timer в отдельном worker."""\n    resource.setrlimit(resource.RLIMIT_CORE, (0, 0))\n    if sys.platform.startswith("linux") and hasattr(resource, "RLIMIT_AS"):\n        soft, hard = resource.getrlimit(resource.RLIMIT_AS)\n        value = (\n            memory_bytes if hard == resource.RLIM_INFINITY else min(memory_bytes, hard)\n        )\n        resource.setrlimit(resource.RLIMIT_AS, (value, value))\n    signal.signal(\n        signal.SIGALRM, lambda signum, frame: (_ for _ in ()).throw(ProgramTimeout())\n    )\n\n\ndef prepare_pristine_worker():\n    """Run once in a Torch-free, single-threaded parent; never run user code here."""\n    _MODULES.clear()\n    safe_import("math")\n    safe_import("sympy")\n    random.seed(0)\n\n\ndef execute_task(payload, tree=None, isolated=False):\n    """Новый env на задачу, wall/CPU timers, только конечный numeric solve."""\n    code, wall_limit, cpu_limit = payload\n    if not isolated:\n        _MODULES.clear()\n    random.seed(0)\n    if tree is None:\n        tree, status = filter_program(code)\n        if status:\n            return {"status": status}\n    used = resource.getrusage(resource.RUSAGE_SELF)\n    soft = max(1, math.ceil(used.ru_utime + used.ru_stime) + int(math.ceil(cpu_limit)))\n    _, hard = resource.getrlimit(resource.RLIMIT_CPU)\n    if hard != resource.RLIM_INFINITY:\n        soft = min(soft, hard)\n    resource.setrlimit(resource.RLIMIT_CPU, (soft, hard))\n    signal.signal(signal.SIGPROF, lambda *_: (_ for _ in ()).throw(ProgramTimeout()))\n    signal.setitimer(signal.ITIMER_PROF, cpu_limit)\n    signal.setitimer(signal.ITIMER_REAL, wall_limit)\n    try:\n        env = {"__builtins__": dict(SAFE_BUILTINS, __import__=safe_import)}\n        exec(compile(tree, "<bounded-dataset-program>", "exec"), env, env)\n        solve = env.get("solve")\n        if not callable(solve):\n            return {"status": "missing_solve"}\n        value = solve()\n        if isinstance(value, bool):\n            value = int(value)\n        if not isinstance(value, (int, float)):\n            if type(value).__module__.startswith("sympy") and value.is_real is True:\n                value = float(value)\n            else:\n                return {"status": "non_number"}\n        value = float(value)\n        if not math.isfinite(value):\n            return {"status": "non_finite"}\n        return {"status": "numeric", "value": value}\n    except ProgramTimeout:\n        return {"status": "timeout"}\n    except BaseException as e:\n        return {"status": "execution_error", "error_type": type(e).__name__}\n    finally:\n        signal.setitimer(signal.ITIMER_REAL, 0)\n        signal.setitimer(signal.ITIMER_PROF, 0)\n', '_qk_attention.py': '"""Совместимый импорт; архитектура полностью описана в model.py."""\nfrom .model import HeadLayerNorm, QKAttention, install_qk_norm, norm_q, norm_k\n', '_service.py': '"""Torch-free очередь с заменой зависших workers и расширением 2 → 4."""\n\nimport json\nimport multiprocessing as mp\nimport os\nimport queue\nimport signal\nimport sys\nimport threading\nimport time\nfrom collections import deque\nfrom multiprocessing.connection import wait\n\nfrom ._worker import WALL, worker\n\n\ndef lower_priority():\n    error = None\n    try:\n        effective = os.nice(10)\n    except PermissionError as exc:\n        effective, error = os.getpriority(os.PRIO_PROCESS, 0), str(exc)\n    return {\n        "nice_requested_increment": 10,\n        "nice": effective,\n        "nice_applied": error is None,\n        "nice_error": error,\n    }\n\n\ndef serve(workers, priority):\n    inbox = queue.Queue()\n\n    def read():\n        for line in sys.stdin:\n            inbox.put(json.loads(line))\n        inbox.put(None)\n\n    threading.Thread(target=read, daemon=True).start()\n    context = mp.get_context("spawn")\n    slots = []\n    pending = deque()\n    finished = False\n\n    def start():\n        parent, child = context.Pipe()\n        process = context.Process(target=worker, args=(child,), daemon=True)\n        process.start()\n        child.close()\n        return [process, parent, None, 0.0]\n\n    def stop(slot):\n        process, pipe, _, _ = slot\n        if process.is_alive():\n            process.kill()\n        process.join(timeout=0.05)\n        pipe.close()\n\n    def emit(index, result, metrics=None):\n        print(json.dumps([index, result, metrics]), flush=True)\n\n    try:\n        slots = [start() for _ in range(workers)]\n\n        def state():\n            print(json.dumps({"workers": len(slots), **priority}), flush=True)\n\n        state()\n        while not finished or pending or any(s[2] is not None for s in slots):\n            while True:\n                try:\n                    batch = inbox.get_nowait()\n                except queue.Empty:\n                    break\n                if batch is None:\n                    finished = True\n                elif isinstance(batch, dict):\n                    target = batch["workers"]\n                    if not isinstance(target, int) or not len(slots) <= target <= 4:\n                        raise ValueError(\n                            "Only expansion up to four workers is supported"\n                        )\n                    slots.extend(start() for _ in range(target - len(slots)))\n                    state()\n                else:\n                    pending.extend(batch)\n            for number, slot in enumerate(slots):\n                process, pipe, active, submitted = slot\n                pipe_failed = False\n                # Worker enforces wall+2; allow a bounded IPC/reap margin here.\n                try:\n                    if active is not None and pipe.poll():\n                        index, result, *metadata = pipe.recv()\n                        assert index == active\n                        emit(index, result, metadata[0] if metadata else None)\n                        slot[2] = None\n                except (EOFError, OSError):\n                    pipe_failed = True\n                if (\n                    pipe_failed\n                    or not process.is_alive()\n                    or (\n                        slot[2] is not None\n                        and time.monotonic() - submitted >= WALL + 2.5\n                    )\n                ):\n                    if slot[2] is not None:\n                        emit(slot[2], {"status": "timeout_or_worker_exit"})\n                    stop(slot)\n                    slots[number] = slot = start()\n                    process, pipe = slot[:2]\n                if slot[2] is None and pending:\n                    index, code = pending.popleft()\n                    try:\n                        pipe.send((index, code))\n                        slot[2:] = [index, time.monotonic()]\n                    except (EOFError, OSError):\n                        # Death can race is_alive(): this job was not accepted.\n                        pending.appendleft((index, code))\n                        stop(slot)\n                        slots[number] = start()\n            active_pipes = [slot[1] for slot in slots if slot[2] is not None]\n            if active_pipes:\n                wait(\n                    active_pipes, timeout=0.002\n                )  # Wake immediately, no per-program 1ms polling tax.\n            else:\n                time.sleep(0.002)\n    finally:\n        for slot in slots:\n            stop(slot)\n\n\nif __name__ == "__main__":\n    if sys.platform.startswith("linux"):\n        import ctypes\n\n        parent = os.getppid()\n        signal.signal(\n            signal.SIGTERM, lambda *_: os.killpg(os.getpgrp(), signal.SIGKILL)\n        )\n        assert ctypes.CDLL(None).prctl(1, signal.SIGTERM, 0, 0, 0) == 0\n        if parent == 1 or os.getppid() != parent:\n            os.killpg(os.getpgrp(), signal.SIGKILL)\n    serve(int(sys.argv[1]), lower_priority())\n', '_worker.py': '"""Изолированное исполнение: fresh env для math, pristine fork для SymPy."""\n\nimport json\nimport os\nimport resource\nimport select\nimport signal\nimport sys\nimport time\n\nfrom . import _numeric as numeric\n\nMEMORY = 768 * 1024**2\nWALL, CPU = 3.0, 2.0\n\n\ndef kill_program(pid):\n    try:\n        os.killpg(pid, signal.SIGKILL)\n    except (ProcessLookupError, PermissionError):\n        try:\n            os.kill(pid, signal.SIGKILL)\n        except ProcessLookupError:\n            pass\n    until = time.monotonic() + 0.1\n    while time.monotonic() < until:\n        try:\n            if os.waitpid(pid, os.WNOHANG)[0]:\n                return\n        except ChildProcessError:\n            return\n        time.sleep(0.001)\n    raise RuntimeError("Program could not be reaped; replace pristine worker")\n\n\ndef isolated_task(code, tree, connection):\n    reader, writer = os.pipe()\n    parent = os.getpid()\n    pid = os.fork()\n    if pid == 0:\n        try:\n            os.close(reader)\n            connection.close()\n            os.setsid()\n            if sys.platform.startswith("linux"):\n                import ctypes\n\n                signal.signal(\n                    signal.SIGTERM, lambda *_: os.killpg(os.getpgrp(), signal.SIGKILL)\n                )\n                if (\n                    ctypes.CDLL(None).prctl(1, signal.SIGTERM, 0, 0, 0) != 0\n                    or os.getppid() != parent\n                ):\n                    os.killpg(os.getpgrp(), signal.SIGKILL)\n            resource.setrlimit(resource.RLIMIT_CPU, (int(CPU) + 1, int(CPU) + 1))\n            result = numeric.execute_task((code, WALL, CPU), tree=tree, isolated=True)\n            os.write(writer, json.dumps(result, separators=(",", ":")).encode())\n        finally:\n            os._exit(0)\n    os.close(writer)\n    try:\n        if select.select([reader], [], [], WALL + 2.0)[0]:\n            data = os.read(reader, 4096)\n            if data:\n                return json.loads(data)\n        return {"status": "timeout_or_worker_exit"}\n    finally:\n        os.close(reader)\n        kill_program(pid)\n\n\ndef worker(connection):\n    with open(os.devnull, "w") as sink:\n        os.dup2(sink.fileno(), 1)\n        os.dup2(sink.fileno(), 2)\n    numeric.worker_init(MEMORY)\n    pristine = None\n    while True:\n        item = connection.recv()\n        if item is None:\n            return\n        index, code = item\n        started = time.monotonic()\n        used = time.process_time()\n        before = resource.getrusage(resource.RUSAGE_CHILDREN)\n        tree, status = numeric.filter_program(code)\n        if status:\n            result = {"status": status}\n        elif any(\n            isinstance(node, numeric.ast.Import)\n            and any(a.name == "sympy" for a in node.names)\n            or (isinstance(node, numeric.ast.ImportFrom) and node.module == "sympy")\n            for node in numeric.ast.walk(tree)\n        ):\n            if pristine is None:\n                numeric.prepare_pristine_worker()\n                pristine = dict(numeric._MODULES)\n            numeric._MODULES.clear()\n            numeric._MODULES.update(pristine)\n            result = isolated_task(code, tree, connection)\n        else:\n            result = numeric.execute_task((code, WALL, CPU), tree=tree)\n        after = resource.getrusage(resource.RUSAGE_CHILDREN)\n        connection.send(\n            (\n                index,\n                result,\n                {\n                    "task_wall_seconds": time.monotonic() - started,\n                    "task_cpu_seconds": time.process_time()\n                    - used\n                    + after.ru_utime\n                    + after.ru_stime\n                    - before.ru_utime\n                    - before.ru_stime,\n                },\n            )\n        )\n', 'config.json': '{\n  "models": [\n    {\n      "name": "C-long2x-B6",\n      "filename": "c-long2x-b6-fp16.pt",\n      "sha256": "ee30794048bb6200cf97ac4d7e5c1606f0e97893ea04f6567c9487f6c04345fa",\n      "dataset": "friend1/nlp-hw02-b6",\n      "source_kind": "notebooks",\n      "source_ref": "friend1/nlp-hw02-b6/1",\n      "K1": 1,\n      "K2": 1\n    },\n    {\n      "name": "pilot-b",\n      "filename": "p1-b-fp16.pt",\n      "sha256": "8439037720dda16dda6b73fe676860d5498bbcba4cacebc0d709bbb22db464a5",\n      "dataset": "friend1/nlp-hw02-kaggle-p1",\n      "source_kind": "notebooks",\n      "source_ref": "friend1/nlp-hw02-kaggle-p1/3",\n      "K1": 1,\n      "K2": 8\n    },\n    {\n      "name": "W-B6",\n      "filename": "w-b6-fp16.pt",\n      "sha256": "cb34c81e6cda83c73bf4662c5c425446be8cdb48c30dd5e9e217fd6cbc1cd496",\n      "dataset": "friend1/nlp-hw02-b6",\n      "source_kind": "notebooks",\n      "source_ref": "friend1/nlp-hw02-b6/1",\n      "K1": 1,\n      "K2": 8\n    }\n  ],\n  "unanimity_threshold": 1.0,\n  "sequence_budget": 640,\n  "greedy_sequence_budget": 640,\n  "scorer_enabled": false,\n  "scorer_weight": 0.0,\n  "device": "cuda:0",\n  "budget_seconds": 180.0,\n  "cap": 512,\n  "seed": 270042,\n  "temperature": 0.7,\n  "tail_workers": 4,\n  "generation_workers": 2,\n  "expected_tune_matches": 1547,\n  "tokenizer_sha256": "261f25b5d4be006a6a00e41e0eafbe522580741da627ba670f6b9805a2659b7d",\n  "decoder": {\n    "share_cross": true,\n    "async_eos": true,\n    "eos_every": 32,\n    "compact_done": true,\n    "compact_fraction": 0.25,\n    "race_sampler": true,\n    "lean_greedy": true\n  },\n  "guards": {\n    "generation": 160.0,\n    "execution": 166.0,\n    "verification": 170.0,\n    "supervisor": 172.0,\n    "greedy": 130.0\n  },\n  "cuda_allocator": "expandable_segments:True,garbage_collection_threshold:0.8",\n  "release_cache_between_batches": true\n}\n', 'config.py': '"""Единственный файл политики — config.json; пути задаются вызывающим notebook."""\n\nimport json\nfrom pathlib import Path\n\n\ndef default_config():\n    """Вернуть независимую копию замороженной политики v11 с budget 768."""\n    return json.loads(Path(__file__).with_name("config.json").read_text())\n', 'data.py': '"""Чтение sealed compact-кеша и точный порядок исходного обучения L1/L2.\n\nИсточник: только исходные пары tinysms_qa_12m. Целевые строки не синтезируются.\nПолная подготовка корпуса сохранена в work (26a/26c/L1 cache).\n"""\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nORDER_SEED = 2026101201\n\n\nclass CompactCache:\n    """Int16 content + int64 offsets; mmap не загружает весь корпус в RAM.\n\n    Совместим с compact-manifest.json боевого L1; source уже содержит BOS/EOS.\n    """\n\n    def __init__(self, root):\n        self.root = Path(root)\n        manifest = json.loads((self.root / "compact-manifest.json").read_text())\n        if not manifest["sealed"] or manifest["representation"] != 30:\n            raise ValueError("Expected sealed compact representation 30")\n        self.parts, starts, sl, tl = [], [0], [], []\n        for shard in manifest["shards"]:\n            folder = self.root / shard["folder"]\n            arrays = {\n                name: np.load(\n                    folder / (name + ".npy"), mmap_mode="r", allow_pickle=False\n                )\n                for name in ("source", "target", "source-offsets", "target-offsets")\n            }\n            for prefix in ("source", "target"):\n                tokens, offsets = arrays[prefix], arrays[prefix + "-offsets"]\n                if tokens.dtype != np.int16 or offsets.dtype != np.int64:\n                    raise ValueError("Expected int16 tokens and int64 offsets")\n                lengths = np.diff(offsets)\n                if (\n                    offsets[0] != 0\n                    or offsets[-1] != len(tokens)\n                    or np.any(lengths <= 0)\n                ):\n                    raise ValueError("Invalid cache offsets")\n                if lengths.max() + (prefix == "target") > 2048:\n                    raise ValueError("Position overflow")\n            n = len(arrays["source-offsets"]) - 1\n            if n != shard["rows"] or n != len(arrays["target-offsets"]) - 1:\n                raise ValueError("Source/target row mismatch")\n            self.parts.append(arrays)\n            starts.append(starts[-1] + n)\n            sl.append(np.diff(arrays["source-offsets"]).astype(np.uint16))\n            tl.append(np.diff(arrays["target-offsets"]).astype(np.uint16))\n        self.starts = np.asarray(starts)\n        self.source_lengths, self.target_lengths = (\n            np.concatenate(sl),\n            np.concatenate(tl),\n        )\n        if len(self) != manifest["rows"]:\n            raise ValueError("Manifest row mismatch")\n\n    def __len__(self):\n        return int(self.starts[-1])\n\n    def __getitem__(self, index):\n        index = int(index)\n        if not 0 <= index < len(self):\n            raise IndexError(index)\n        shard = int(np.searchsorted(self.starts, index, side="right") - 1)\n        local = index - int(self.starts[shard])\n        arrays = self.parts[shard]\n        return tuple(\n            arrays[p][arrays[p + "-offsets"][local] : arrays[p + "-offsets"][local + 1]]\n            for p in ("source", "target")\n        )\n\n\ndef collate(cache, indices, device):\n    """Right padding: src=BOS+Q+EOS, tgt=BOS+compact, labels=compact+EOS."""\n    import torch\n\n    pairs = [cache[int(i)] for i in indices]\n    ns = [len(s) for s, t in pairs]\n    nt = [len(t) for s, t in pairs]\n    src = np.zeros((len(pairs), max(ns)), dtype=np.int64)\n    tgt = np.zeros((len(pairs), max(nt) + 1), dtype=np.int64)\n    labels = np.zeros_like(tgt)\n    for j, (s, t) in enumerate(pairs):\n        src[j, : len(s)] = s\n        tgt[j, 0] = 1\n        tgt[j, 1 : len(t) + 1] = t\n        labels[j, : len(t)] = t\n        labels[j, len(t)] = 2\n    counts = {\n        "pairs": len(pairs),\n        "source_tokens": sum(ns),\n        "target_tokens": sum(n + 1 for n in nt),\n        "source_slots": src.size,\n        "target_slots": tgt.size,\n    }\n    return (\n        torch.from_numpy(src).to(device),\n        torch.from_numpy(tgt).to(device),\n        torch.from_numpy(labels).to(device),\n        counts,\n    )\n\n\ndef make_order(cache, seed=ORDER_SEED, batch=128, chunk=4096):\n    rng = np.random.default_rng(seed)\n    order = rng.permutation(len(cache)).astype(np.uint32)\n    for lo in range(0, len(order), chunk):\n        part = order[lo : lo + chunk]\n        ns = cache.source_lengths[part].astype(np.int32)\n        nt = cache.target_lengths[part].astype(np.int32)\n        part = part[np.lexsort((nt, ns, ns + nt + 1))]\n        full = len(part) // batch\n        if full:\n            part[: full * batch] = (\n                part[: full * batch]\n                .reshape(full, batch)[rng.permutation(full)]\n                .reshape(-1)\n            )\n        order[lo : lo + len(part)] = part\n    assert np.array_equal(np.sort(order), np.arange(len(cache), dtype=np.uint32))\n    return order\n\n\ndef threshold_for(lengths, fraction):\n    """Nearest attainable >=threshold fraction; ties choose the higher threshold."""\n    counts = np.bincount(np.asarray(lengths, dtype=np.int64))\n    tails = np.cumsum(counts[::-1])[::-1]\n    options = [\n        (abs(int(tails[t]) / len(lengths) - fraction), -t, t)\n        for t in np.flatnonzero(counts)\n    ]\n    options.append((fraction, -len(counts), len(counts)))\n    return min(options)[2]\n\n\ndef long2x_order(\n    cache,\n    heldout_lengths,\n    original_lengths=None,\n    seed=2026101302,\n    batch=128,\n    chunk=4096,\n):\n    """Каждая длинная строка встречается дважды; порог по доле heldout >=90.\n\n    original_lengths — исходные raw-BPE длины, выровненные с compact-кешем.\n    Без них используется compact analogue порога, как в L2 fallback.\n    """\n    lengths = (\n        cache.target_lengths\n        if original_lengths is None\n        else np.asarray(original_lengths)\n    )\n    fraction = float(np.mean(np.asarray(heldout_lengths) >= 90))\n    threshold = threshold_for(lengths, fraction)\n    slots = np.concatenate(\n        (\n            np.arange(len(cache), dtype=np.uint32),\n            np.flatnonzero(lengths >= threshold).astype(np.uint32),\n        )\n    )\n    rng = np.random.default_rng(seed)\n    rng.shuffle(slots)\n    for lo in range(0, len(slots), chunk):\n        part = slots[lo : lo + chunk]\n        ns = cache.source_lengths[part].astype(np.int32)\n        nt = cache.target_lengths[part].astype(np.int32)\n        part = part[np.lexsort((nt, ns, ns + nt + 1))]\n        full = len(part) // batch\n        if full:\n            part[: full * batch] = (\n                part[: full * batch]\n                .reshape(full, batch)[rng.permutation(full)]\n                .reshape(-1)\n            )\n        slots[lo : lo + len(part)] = part\n    return slots\n\n\ndef take(order, cursor, batch):\n    """Продолжить ту же перестановку циклически; cursor считает предъявления."""\n    return np.asarray(order[(np.arange(batch, dtype=np.int64) + cursor) % len(order)])\n', 'decode.py': '"""Быстрый v11 decoder: KV-cache, compaction, asynchronous EOS и Exp(1) sampling.\n\nCompaction и sampling намеренно сохраняют порядок операций final_speed.py.\nОдинаковые seed, batch, precision и runtime нужны для воспроизводимых samples.\n"""\n\nimport math\nimport time\nfrom dataclasses import dataclass\n\nimport torch\nfrom torch.nn import functional as F\n\nfrom . import model as C\n\n\nclass BudgetExpired(Exception):\n    """Новые batches больше не начинаются; готовые кандидаты сохраняются."""\n\n\ndef check_deadline(deadline):\n    if deadline is not None and time.monotonic() >= deadline:\n        raise BudgetExpired("Inference deadline reached")\n\n\nclass Context:\n    """Encoder и неизменные cross-attention K/V вычисляются один раз на batch."""\n\n    @torch.inference_mode()\n    def __init__(self, model, src, capacity):\n        self.model, self.src, self.capacity = (model, src, capacity)\n        self.width = model.width\n        self.heads = model.transformer.decoder.layers[0].self_attn.num_heads\n        self.dim = self.width // self.heads\n        memory = model.transformer.encoder(\n            model.embed(src), src_key_padding_mask=src.eq(C.PAD)\n        )\n        self.allowed = src.ne(C.PAD)[:, None, None, :]\n        self.positions = model.positions(torch.arange(capacity, device=src.device))\n        self.cross = []\n        for layer in model.transformer.decoder.layers:\n            a = layer.multihead_attn\n            k, v = F.linear(\n                memory, a.in_proj_weight[self.width :], a.in_proj_bias[self.width :]\n            ).chunk(2, -1)\n            kh = (\n                k.reshape(len(src), -1, self.heads, self.dim)\n                .transpose(1, 2)\n                .contiguous()\n            )\n            if hasattr(a, "k_norm"):\n                kh = a.k_norm(kh)\n            self.cross.append(\n                (\n                    kh,\n                    v.reshape(len(src), -1, self.heads, self.dim)\n                    .transpose(1, 2)\n                    .contiguous(),\n                )\n            )\n\n\nclass BaseDecoder:\n    """Один autoregressive шаг; self-attention K/V предвыделены до cap."""\n\n    @torch.inference_mode()\n    def __init__(self, context, k):\n        self.c, self.k, self.t = (context, k, 0)\n        self.n = len(context.src) * k\n        self.allowed = context.allowed.repeat_interleave(k, dim=0)\n        self.cross = [\n            tuple(t.repeat_interleave(k, dim=0).contiguous() for t in pair)\n            for pair in context.cross\n        ]\n        self.cross_bytes = sum(\n            t.numel() * t.element_size() for pair in self.cross for t in pair\n        )\n        shape = (self.n, context.heads, context.capacity, context.dim)\n        self.self_kv = [\n            (context.positions.new_empty(shape), context.positions.new_empty(shape))\n            for _ in context.cross\n        ]\n        self.valid = torch.zeros(\n            self.n, context.capacity, dtype=torch.bool, device=context.src.device\n        )\n\n    def heads(self, x):\n        return x.reshape(self.n, 1, self.c.heads, self.c.dim).transpose(1, 2)\n\n    def attend(self, a, q, k, v, mask):\n        assert q.ndim == k.ndim == v.ndim == 4\n        if hasattr(a, "q_norm"):\n            q = a.q_norm(q)\n        y = F.scaled_dot_product_attention(\n            q, k, v, attn_mask=mask, dropout_p=0.0, is_causal=False\n        )\n        return a.out_proj(\n            y.transpose(1, 2).contiguous().reshape(self.n, 1, self.c.width)\n        )\n\n    @torch.inference_mode()\n    def step(self, tokens):\n        c = self.c\n        t = self.t\n        x = (\n            c.model.tokens(tokens[:, None]) * math.sqrt(c.width)\n            + c.positions[t][None, None, :]\n        )\n        self.valid[:, t] = tokens.ne(C.PAD)\n        for layer, (sk, sv), (ck, cv) in zip(\n            c.model.transformer.decoder.layers, self.self_kv, self.cross\n        ):\n            a = layer.self_attn\n            q, k, v = F.linear(layer.norm1(x), a.in_proj_weight, a.in_proj_bias).chunk(\n                3, -1\n            )\n            sk[:, :, t : t + 1] = (\n                a.k_norm(self.heads(k)) if hasattr(a, "k_norm") else self.heads(k)\n            )\n            sv[:, :, t : t + 1] = self.heads(v)\n            x = x + layer.dropout1(\n                self.attend(\n                    a,\n                    self.heads(q),\n                    sk[:, :, : t + 1],\n                    sv[:, :, : t + 1],\n                    self.valid[:, None, None, : t + 1],\n                )\n            )\n            a = layer.multihead_attn\n            q = F.linear(\n                layer.norm2(x), a.in_proj_weight[: c.width], a.in_proj_bias[: c.width]\n            )\n            x = x + layer.dropout2(self.attend(a, self.heads(q), ck, cv, self.allowed))\n            x = x + layer.dropout3(\n                layer.linear2(\n                    layer.dropout(layer.activation(layer.linear1(layer.norm3(x))))\n                )\n            )\n        if c.model.transformer.decoder.norm is not None:\n            x = c.model.transformer.decoder.norm(x)\n        self.t += 1\n        return c.model.head(x[:, 0])\n\n\n@dataclass(frozen=True)\nclass Options:\n    """Настройки decoder all, использованные в v11."""\n\n    share_cross: bool = True\n    async_eos: bool = True\n    eos_every: int = 32\n    compact_done: bool = True\n    compact_fraction: float = 0.25\n    race_sampler: bool = True\n    lean_greedy: bool = True\n\n\nclass Decoder(BaseDecoder):\n    @torch.inference_mode()\n    def __init__(self, context, k, share_cross=False):\n        if not share_cross or k != 1:\n            super().__init__(context, k)\n            return\n        self.c, self.k, self.t, self.n = (context, k, 0, len(context.src))\n        self.allowed, self.cross = (context.allowed, context.cross)\n        self.cross_bytes = sum(\n            t.numel() * t.element_size() for pair in self.cross for t in pair\n        )\n        shape = (self.n, context.heads, context.capacity, context.dim)\n        self.self_kv = [\n            (context.positions.new_empty(shape), context.positions.new_empty(shape))\n            for _ in context.cross\n        ]\n        self.valid = torch.zeros(\n            self.n, context.capacity, dtype=torch.bool, device=context.src.device\n        )\n\n    @torch.inference_mode()\n    def select(self, index):\n        """Copy only initialized self-KV prefix, then continue at the SAME position."""\n        if index.ndim != 1 or not len(index):\n            raise ValueError("Nonempty 1-D active indices required")\n        n, cap = (len(index), self.c.capacity)\n        self.allowed = self.allowed.index_select(0, index)\n        self.cross = [\n            tuple(t.index_select(0, index) for t in pair) for pair in self.cross\n        ]\n        for number in range(len(self.self_kv)):\n            pair = self.self_kv[number]\n            new_pair = []\n            for old in pair:\n                new = old.new_empty((n, self.c.heads, cap, self.c.dim))\n                new[:, :, : self.t].copy_(old[:, :, : self.t].index_select(0, index))\n                new_pair.append(new)\n            self.self_kv[number] = tuple(new_pair)\n        self.n = n\n        self.valid = self.valid.index_select(0, index)\n\n\ndef race_pick(probabilities, noise):\n    """Independent Exp(1) race: argmax(p/E) has categorical probabilities p.\n\n    Uses exactly the same FP32 softmax as baseline; RNG consumption can change.\n    Avoids multinomial\'s probability validation/reductions. Noise storage is reused.\n    """\n    noise.exponential_()\n    return torch.div(probabilities, noise, out=noise).argmax(-1)\n\n\n@torch.inference_mode()\ndef decode(\n    context,\n    k=1,\n    temperature=None,\n    deadline=None,\n    compute_logprob=True,\n    options=None,\n    stats=None,\n):\n    options = options or Options()\n    d = Decoder(context, k, options.share_cross)\n    device, n = (context.src.device, d.n)\n    token = torch.full((n,), C.BOS, device=device, dtype=torch.long)\n    done = torch.zeros(n, dtype=torch.bool, device=device)\n    output = torch.full((n, context.capacity), C.PAD, device=device, dtype=torch.long)\n    sums, lengths = (\n        torch.zeros(n, device=device),\n        torch.zeros(n, dtype=torch.long, device=device),\n    )\n    final_sums, final_lengths = (torch.zeros_like(sums), torch.zeros_like(lengths))\n    original = torch.arange(n, device=device)\n    compacted = False\n    noise = None\n    host = event = None\n    pending = False\n    async_eos = options.async_eos and device.type == "cuda"\n    if async_eos:\n        host = torch.empty(n, dtype=torch.bool, pin_memory=True)\n        event = torch.cuda.Event()\n    # One temperature per stochastic slot, shared across questions. Compaction\n    # must retain each sequence\'s original slot temperature.\n    temperatures = None\n    if isinstance(temperature, (list, tuple)):\n        if len(temperature) != k or any(not math.isfinite(t) or t <= 0 for t in temperature):\n            raise ValueError("Require K positive finite slot temperatures")\n        temperatures = torch.tensor(temperature, device=device, dtype=torch.float32).repeat(len(context.src))[:, None]\n    history = []\n    checks = 0\n\n    def flush():\n        final_sums.index_copy_(0, original, sums)\n        final_lengths.index_copy_(0, original, lengths)\n\n    def inspect(snapshot):\n        nonlocal token, done, sums, lengths, original, compacted, noise, host, temperatures\n        if bool(snapshot.all()):\n            return True\n        if not options.compact_done:\n            return False\n        keep = (~snapshot).nonzero().flatten()\n        if options.compact_done and len(keep) <= d.n * (1 - options.compact_fraction):\n            flush()\n            index = keep.to(device)\n            before = d.n\n            d.select(index)\n            token, done = (token[index], done[index])\n            sums, lengths, original = (sums[index], lengths[index], original[index])\n            if temperatures is not None:\n                temperatures = temperatures[index]\n            compacted, noise = (True, None)\n            history.append({"step": d.t, "before": before, "after": d.n})\n            if async_eos:\n                host = torch.empty(d.n, dtype=torch.bool, pin_memory=True)\n        return False\n\n    try:\n        for t in range(context.capacity):\n            if t % 16 == 0:\n                check_deadline(deadline)\n            if pending and event.query():\n                pending = False\n                if inspect(host):\n                    break\n            logits = d.step(token)\n            if temperature is not None or compute_logprob:\n                logits = logits.float()\n            logits[:, :2] = -torch.inf\n            if temperature is None:\n                pick = logits.argmax(-1)\n            else:\n                probabilities = F.softmax(logits / (temperatures if temperatures is not None else temperature), -1)\n                if options.race_sampler:\n                    if noise is None:\n                        noise = torch.empty_like(probabilities)\n                    pick = race_pick(probabilities, noise)\n                else:\n                    pick = torch.multinomial(probabilities, 1).squeeze(-1)\n            if compute_logprob:\n                lp = F.log_softmax(logits, -1).gather(-1, pick[:, None]).squeeze(-1)\n                sums.add_(torch.where(done, 0.0, lp))\n            if compute_logprob or not options.lean_greedy:\n                lengths.add_(~done)\n            token = pick.masked_fill(done, C.PAD)\n            if compacted:\n                output[:, t].index_copy_(0, original, token)\n            else:\n                output[:, t] = token\n            done |= token.eq(C.EOS)\n            if (t + 1) % options.eos_every == 0:\n                if async_eos:\n                    if not pending:\n                        host.copy_(done, non_blocking=True)\n                        event.record()\n                        pending = True\n                        checks += 1\n                else:\n                    checks += 1\n                    if inspect(done.cpu()):\n                        break\n    finally:\n        if pending:\n            event.synchronize()\n    flush()\n    if stats is not None:\n        stats.update(\n            steps=d.t,\n            eos_checks=checks,\n            compactions=history,\n            async_eos=async_eos,\n            rng_stream_may_change=options.race_sampler or options.compact_done,\n        )\n    return (\n        output[:, : d.t].reshape(len(context.src), k, d.t).cpu(),\n        (final_sums / final_lengths.clamp_min(1)).reshape(-1, k).cpu(),\n        d.cross_bytes,\n    )\n\n\ndef question_batch(sequence_budget, k):\n    """Лимит задан в последовательностях: batch вопросов = budget // K."""\n    if not isinstance(k, int) or k < 1 or sequence_budget < k:\n        raise ValueError("Require sequence_budget >= K >= 1")\n    return sequence_budget // k\n', 'dry_run.py': '"""Восемь синтетических вопросов и случайные tiny-модели для CPU smoke.\n\nНе оценивает качество и не заменяет cold T4 замер на test1919.\n"""\n\nimport torch\nfrom tokenizers import Tokenizer, models, pre_tokenizers\n\nfrom .model import from_spec\nfrom .tokenizer_repr import MODES\n\n\ndef fixtures():\n    vocabulary = [\n        "[UNK]",\n        *MODES.values(),\n        "How",\n        "many",\n        "is",\n        "plus",\n        "?",\n        *map(str, range(12)),\n        "def solve() -> int:\\n    return 1\\n",\n    ]\n    tok = Tokenizer(\n        models.WordLevel({v: i for i, v in enumerate(vocabulary)}, unk_token="[UNK]")\n    )\n    tok.pre_tokenizer = pre_tokenizers.Whitespace()\n    table = [b"", b"", b""] + [word.encode() for word in vocabulary]\n    spec = dict(\n        d_model=16,\n        heads=2,\n        encoder_layers=1,\n        decoder_layers=1,\n        swiglu_dim=24,\n        vocab=len(table),\n        max_positions=64,\n    )\n    torch.manual_seed(42)\n    models_by_name = {name: from_spec(spec).eval() for name in ("tiny-a", "tiny-b")}\n    rows = [dict(question=f"How many is {i} plus 1 ?") for i in range(8)]\n    specs = [dict(name=name, K1=1, K2=2) for name in models_by_name]\n    return rows, models_by_name, tok, table, specs\n', 'execute.py': '"""Асинхронный CPU executor: дедупликация исполнения без потери голосов.\n\nAST-фильтр, wall 3s / CPU 2s / Linux AS 768 MiB, внешнее kill/reap и\nновые namespaces ограничивают программы датасета. Это не OS sandbox для\nпроизвольного враждебного Python; сеть/файлы недоступны через allowed API.\n"""\n\nimport json\nimport math\nimport os\nimport signal\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\n\n\nclass Executor:\n    """Фоновый сервис; submit не блокирует GPU, drain ограничен deadline."""\n\n    def __init__(self, workers=None):\n        self.workers = workers if workers is not None else 2\n        if (\n            isinstance(self.workers, bool)\n            or not isinstance(self.workers, int)\n            or (not 1 <= self.workers <= 4)\n        ):\n            raise ValueError("Require one to four execution workers")\n        env = dict(\n            os.environ,\n            PYTHONPATH=str(Path(__file__).resolve().parent.parent)\n            + os.pathsep\n            + os.environ.get("PYTHONPATH", ""),\n            OMP_NUM_THREADS="1",\n            MKL_NUM_THREADS="1",\n            OPENBLAS_NUM_THREADS="1",\n            PYTHONHASHSEED="0",\n        )\n        self.child = subprocess.Popen(\n            [sys.executable, "-m", "hw2_codegen._service", str(self.workers)],\n            stdin=subprocess.PIPE,\n            stdout=subprocess.PIPE,\n            stderr=subprocess.DEVNULL,\n            text=True,\n            bufsize=1,\n            env=env,\n            start_new_session=True,\n        )\n        self.service_state = {}\n        self.codes = {}\n        self.results = {}\n        self.error = None\n\n        def read():\n            try:\n                for line in self.child.stdout:\n                    message = json.loads(line)\n                    if isinstance(message, dict):\n                        self.service_state = message\n                        continue\n                    index, value, *_ = message\n                    self.results[index] = value\n            except Exception as error:\n                self.error = type(error).__name__\n\n        self.reader = threading.Thread(target=read, daemon=True)\n        self.reader.start()\n\n    def resize(self, workers):\n        """Добавить workers, сохранив очередь и кеш результатов."""\n        if not isinstance(workers, int) or not self.workers <= workers <= 4:\n            raise ValueError("Only expansion up to four workers is supported")\n        self.child.stdin.write(json.dumps({"workers": workers}) + "\\n")\n        self.child.stdin.flush()\n        self.workers = workers\n\n    def submit(self, candidates):\n        """Послать только новые программы, сохранив отдельные голоса кандидатов."""\n        jobs = []\n        for candidate in candidates:\n            code = candidate["code"]\n            if code not in self.codes:\n                index = len(self.codes)\n                self.codes[code] = index\n                jobs.append([index, code])\n        if jobs:\n            self.child.stdin.write(json.dumps(jobs) + "\\n")\n            self.child.stdin.flush()\n\n    def attach(self, candidates):\n        """Подставить исполнение либо явный budget_not_evaluated."""\n        for candidate in candidates:\n            index = self.codes[candidate["code"]]\n            candidate["execution"] = self.results.get(\n                index, {"status": "budget_not_evaluated"}\n            )\n\n    def drain(self, deadline):\n        """Дождаться результатов и запуска workers до заданного срока."""\n        while (\n            len(self.results) < len(self.codes)\n            or self.service_state.get("workers") != self.workers\n        ) and time.monotonic() < deadline:\n            exit_code = self.child.poll()\n            if exit_code is not None:\n                self.error = self.error or "executor_exit_" + str(exit_code)\n                break\n            if self.error:\n                break\n            time.sleep(0.002)\n        return (\n            len(self.results) == len(self.codes)\n            and self.service_state.get("workers") == self.workers\n        )\n\n    def close(self):\n        try:\n            os.killpg(self.child.pid, signal.SIGKILL)\n        except ProcessLookupError:\n            pass\n        try:\n            self.child.wait(timeout=0.1)\n        except subprocess.TimeoutExpired:\n            pass\n        self.child.stdin.close()\n        self.reader.join(timeout=0.1)\n        self.child.stdout.close()\n\n\ndef verify_winners(winners, deadline, workers=None):\n    """Fresh service: compare the actual chosen CODE against its voted execution."""\n    executor = Executor(workers)\n    eligible = [\n        i for i, w in enumerate(winners) if w["execution"]["status"] == "numeric"\n    ]\n    items = [{"code": winners[i]["code"]} for i in eligible]\n    try:\n        executor.submit(items)\n        complete = executor.drain(deadline)\n        executor.attach(items)\n        mismatches = [\n            i\n            for i, item in zip(eligible, items)\n            if item["execution"]["status"] != "numeric"\n            or not math.isclose(\n                item["execution"]["value"],\n                winners[i]["execution"]["value"],\n                rel_tol=1e-06,\n                abs_tol=1e-06,\n            )\n        ]\n        return {\n            "checked_rows": len(eligible),\n            "complete": complete,\n            "mismatch_rows": mismatches,\n            "completed_unique": len(executor.results),\n        }\n    finally:\n        executor.close()\n', 'model.py': '"""C (6+6/d512) и W (6+6/d768, QK LayerNorm), совместимые с FP16 checkpoints."""\n\nimport math\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nPAD, BOS, EOS, TOKEN_OFFSET = 0, 1, 2, 3\nVOCAB, MAX_POSITIONS, SEED = 8195, 2048, 42\n\n\ndef norm_q(a, x):\n    return a.q_norm(x) if hasattr(a, "q_norm") else x\n\n\ndef norm_k(a, x):\n    return a.k_norm(x) if hasattr(a, "k_norm") else x\n\n\nclass HeadLayerNorm(nn.LayerNorm):\n    """Статистики и affine в FP32, выход в dtype attention."""\n\n    def forward(self, x):\n        # FP32 statistics/affines also when inference casts the entire model to FP16.\n        return F.layer_norm(\n            x.float(),\n            self.normalized_shape,\n            self.weight.float(),\n            self.bias.float(),\n            self.eps,\n        ).to(x.dtype)\n\n\nclass QKAttention(nn.MultiheadAttention):\n    """Headwise LayerNorm Q/K с прежними именами MHA-параметров."""\n\n    def forward(\n        self,\n        query,\n        key,\n        value,\n        key_padding_mask=None,\n        need_weights=True,\n        attn_mask=None,\n        average_attn_weights=True,\n        is_causal=False,\n    ):\n        assert self.batch_first and query.ndim == 3 and self._qkv_same_embed_dim\n        assert not self.add_zero_attn and self.bias_k is None and self.bias_v is None\n        width = self.embed_dim\n        n, t, _ = query.shape\n        length = key.shape[1]\n        dim = width // self.num_heads\n        wq, wk, wv = self.in_proj_weight.chunk(3)\n        bq, bk, bv = (\n            self.in_proj_bias.chunk(3) if self.in_proj_bias is not None else (None,) * 3\n        )\n\n        def heads(x):\n            return x.reshape(n, -1, self.num_heads, dim).transpose(1, 2)\n\n        q = norm_q(self, heads(F.linear(query, wq, bq)))\n        k = norm_k(self, heads(F.linear(key, wk, bk)))\n        v = heads(F.linear(value, wv, bv))\n        mask = None\n        if attn_mask is not None:\n            mask = attn_mask\n            if mask.dtype == torch.bool:\n                mask = torch.zeros_like(mask, dtype=q.dtype).masked_fill(\n                    mask, float("-inf")\n                )\n            if mask.ndim == 2:\n                mask = mask[None, None]\n            elif mask.ndim == 3:\n                mask = mask.reshape(n, self.num_heads, t, length)\n        elif is_causal:\n            mask = torch.full(\n                (t, length), float("-inf"), dtype=q.dtype, device=q.device\n            ).triu(1)[None, None]\n        if key_padding_mask is not None:\n            padding = key_padding_mask\n            if padding.dtype == torch.bool:\n                padding = torch.zeros_like(padding, dtype=q.dtype).masked_fill(\n                    padding, float("-inf")\n                )\n            padding = padding[:, None, None, :]\n            mask = padding if mask is None else mask + padding\n        if getattr(self, "record_logits", False):\n            # Cheap sampled maximum: first batch, all heads, first eight queries, all keys.\n            with torch.no_grad(), torch.autocast(\n                device_type=q.device.type, enabled=False\n            ):\n                sample = (\n                    q[:1, :, :8].float() @ k[:1].float().transpose(-1, -2)\n                ) / math.sqrt(dim)\n                self.sampled_logit_max = sample.abs().amax().detach()\n        p = self.dropout if self.training else 0.0\n        if need_weights:\n            weights = (q @ k.transpose(-1, -2)) / math.sqrt(dim)\n            if mask is not None:\n                weights = weights + mask\n            weights = F.dropout(weights.softmax(-1), p, training=self.training)\n            y = weights @ v\n            returned = weights.mean(1) if average_attn_weights else weights\n        else:\n            y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=p)\n            returned = None\n        return (\n            self.out_proj(y.transpose(1, 2).contiguous().reshape(n, t, width)),\n            returned,\n        )\n\n\ndef install_qk_norm(transformer, flag):\n    """Установить QK-нормировку во всех self/cross attention без смены весов MHA."""\n    if flag in (False, None, "none"):\n        return\n    if flag not in (True, "layernorm"):\n        raise ValueError("qk_norm must be false or layernorm")\n    for layer in [*transformer.encoder.layers, *transformer.decoder.layers]:\n        # Disable PyTorch\'s fused encoder path, which would bypass the QKAttention forward.\n        if isinstance(layer, nn.TransformerEncoderLayer):\n            layer.activation_relu_or_gelu = 0\n        for a in (\n            layer.self_attn,\n            *([layer.multihead_attn] if hasattr(layer, "multihead_attn") else []),\n        ):\n            a.__class__ = QKAttention\n            a.q_norm = HeadLayerNorm(a.head_dim, eps=1e-5)\n            a.k_norm = HeadLayerNorm(a.head_dim, eps=1e-5)\n\n\nclass SwiGLUInput(nn.Module):\n    """Две проекции: SiLU(gate(x)) * value(x)."""\n\n    def __init__(self, width, hidden):\n        super().__init__()\n        self.gate = nn.Linear(width, hidden)\n        self.value = nn.Linear(width, hidden)\n\n    def forward(self, x):\n        return F.silu(self.gate(x)) * self.value(x)\n\n\ndef replace_ffn_with_swiglu(transformer, width, hidden):\n    """SwiGLU вместо FFN; фиксированный локальный RNG исходного рецепта."""\n    with torch.random.fork_rng(devices=[]):\n        torch.set_rng_state(torch.Generator().manual_seed(SEED).get_state())\n        for layer in list(transformer.encoder.layers) + list(\n            transformer.decoder.layers\n        ):\n            layer.linear1 = SwiGLUInput(width, hidden)\n            layer.linear2 = nn.Linear(hidden, width)\n            layer.activation = nn.Identity()\n            if isinstance(layer, nn.TransformerEncoderLayer):\n                layer.activation_relu_or_gelu = 0\n            for module in [layer.linear1.gate, layer.linear1.value, layer.linear2]:\n                nn.init.xavier_uniform_(module.weight)\n\n\nclass SimpleTransformer(nn.Module):\n    """Pre-LN encoder–decoder с learned positions и отдельной выходной головой."""\n\n    def __init__(\n        self,\n        d_model=256,\n        heads=4,\n        encoder_layers=4,\n        decoder_layers=4,\n        swiglu_dim=680,\n        vocab=VOCAB,\n        max_positions=MAX_POSITIONS,\n        dropout=0.1,\n        normalized_init=True,\n        qk_norm=False,\n    ):\n        super().__init__()\n        self.width = d_model\n        self.vocab = vocab\n        self.tokens = nn.Embedding(vocab, d_model, padding_idx=PAD)\n        self.positions = nn.Embedding(max_positions, d_model)\n        self.transformer = nn.Transformer(\n            d_model=d_model,\n            nhead=heads,\n            num_encoder_layers=encoder_layers,\n            num_decoder_layers=decoder_layers,\n            dim_feedforward=4 * d_model,\n            dropout=dropout,\n            batch_first=True,\n            norm_first=True,\n        )\n        self.transformer.encoder.enable_nested_tensor = False\n        self.transformer.encoder.use_nested_tensor = False\n        self.head = nn.Linear(d_model, vocab)\n        replace_ffn_with_swiglu(self.transformer, d_model, swiglu_dim)\n        if qk_norm:\n            install_qk_norm(self.transformer, qk_norm)\n        if normalized_init:\n            with torch.no_grad():\n                self.tokens.weight.div_(math.sqrt(self.width))\n\n    def embed(self, ids):\n        if ids.shape[1] > self.positions.num_embeddings:\n            raise ValueError("Position overflow")\n        return self.tokens(ids) * math.sqrt(self.width) + self.positions(\n            torch.arange(ids.shape[1], device=ids.device)\n        )\n\n    def decode_prefix(self, tgt, memory, source_padding):\n        mask = torch.ones(\n            tgt.shape[1], tgt.shape[1], device=tgt.device, dtype=torch.bool\n        ).triu(1)\n        return self.transformer.decoder(\n            self.embed(tgt),\n            memory,\n            tgt_mask=mask,\n            tgt_key_padding_mask=tgt.eq(PAD),\n            memory_key_padding_mask=source_padding,\n        )\n\n    def forward(self, src, tgt):\n        memory = self.transformer.encoder(\n            self.embed(src), src_key_padding_mask=src.eq(PAD)\n        )\n        return self.head(self.decode_prefix(tgt, memory, src.eq(PAD)))\n\n    @torch.inference_mode()\n    def generate(self, src, max_new_tokens=512):\n        self.eval()\n        memory = self.transformer.encoder(\n            self.embed(src), src_key_padding_mask=src.eq(PAD)\n        )\n        tgt = torch.full((len(src), 1), BOS, device=src.device, dtype=torch.long)\n        done = torch.zeros(len(src), device=src.device, dtype=torch.bool)\n        for _ in range(max_new_tokens):\n            logits = self.head(self.decode_prefix(tgt, memory, src.eq(PAD))[:, -1])\n            logits[:, [PAD, BOS]] = -torch.inf\n            token = logits.argmax(-1).masked_fill(done, PAD)\n            done |= token.eq(EOS)\n            tgt = torch.cat([tgt, token[:, None]], dim=1)\n            if bool(done.all()):\n                break\n        return tgt[:, 1:]\n\n\nMODEL_C = dict(\n    d_model=512,\n    heads=8,\n    encoder_layers=6,\n    decoder_layers=6,\n    swiglu_dim=1368,\n    vocab=VOCAB,\n)\nMODEL_W = dict(\n    d_model=768,\n    heads=12,\n    encoder_layers=6,\n    decoder_layers=6,\n    swiglu_dim=2048,\n    vocab=VOCAB,\n    qk_norm="layernorm",\n)\n\n\ndef from_spec(model_spec, dropout=0.0):\n    """Создать модель по словарю из checkpoint; веса загружаются отдельно."""\n    return SimpleTransformer(**model_spec, dropout=dropout)\n', 'pipeline.py': '"""Адаптивный v11: greedy → K1 всех моделей → priority waves до K2 → vote.\n\nПорядок samples, per-model RNG и operation order совпадают с боевым v11.\nSHA весов проверяется до загрузки; Kaggle API не нужен.\n"""\n\nimport json\nimport hashlib\nimport gzip\nimport math\nimport time\nfrom pathlib import Path\n\nimport torch\nfrom torch.nn.utils.rnn import pad_sequence\n\nfrom .decode import BudgetExpired, Context, check_deadline, decode, question_batch\nfrom .execute import Executor, verify_winners\nfrom .model import PAD, from_spec\nfrom .tokenizer_repr import (\n    active_tokens,\n    decode_bytes,\n    encode_question,\n    restore_compact,\n    tokenizer,\n)\nfrom .vote import candidates, select_available, stage1_decision, stage1_leader_fraction, winner_fraction, slot_temperatures\n\nfrom .config import default_config\nfrom .decode import Options\n\nDEFAULT_MODELS = default_config()["models"]\n\n\ndef sha256(path):\n    """SHA256 файла без загрузки большого веса целиком в RAM."""\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(4 * 1024**2), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef load_model(path, device):\n    """Model-only checkpoint: model_spec + model; FP16 только на CUDA."""\n    payload = torch.load(path, map_location="cpu", weights_only=True)\n    if (\n        payload.get("format") != "nlp-hw02-model-only-v1"\n        or payload.get("optimizer_included") is not False\n    ):\n        raise ValueError("Expected model-only checkpoint without optimizer")\n    model = from_spec(payload["model_spec"]).eval()\n    if device.type == "cuda":\n        model.half()\n    model.load_state_dict(payload["model"], strict=True)\n    return model.to(device)\n\n\ndef pack(ids, lps, table, greedy=False):\n    """Батч token ids → восстановленный solve и средний untempered log p."""\n    groups = []\n    for sequences, means in zip(ids.tolist(), lps.tolist()):\n        group = []\n        for tokens, lp in zip(sequences, means):\n            active = active_tokens(tokens)\n            represented = decode_bytes(table, active).decode("utf8", errors="replace")\n            try:\n                code, error = restore_compact(represented), None\n            except (ValueError, IndexError):\n                code, error = "", "invalid representation"\n            group.append(\n                dict(\n                    code=code,\n                    mean_logprob=0.0 if greedy else (lp if math.isfinite(lp) else None),\n                    represented_code=represented,\n                    restore_error=error,\n                    length=len(active),\n                )\n            )\n        groups.append(group)\n    return groups\n\n\ndef write_answers(path, codes):\n    """Формат бота: одна строка {code: str} на вопрос, исходный порядок."""\n    with Path(path).open("w", encoding="utf8") as stream:\n        for code in codes:\n            if not isinstance(code, str):\n                raise TypeError("Answer code must be a string")\n            stream.write(json.dumps({"code": code}, ensure_ascii=False) + "\\n")\n\n\ndef configure_runtime():\n    """Один CPU thread у Torch оставляет ядра CPU executor; одна CUDA T4."""\n    torch.set_num_threads(1)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.use_deterministic_algorithms(True)\n    torch.utils.deterministic.fill_uninitialized_memory = False\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\n\ndef run_inference(\n    config, *, started=None, rows=None, models=None, tok=None, table=None\n):\n    """Полный timer до записи answers.jsonl; started задаётся ПЕРЕД импортами.\n\n    rows/models/tok/table используются только CPU dry-run и тестами.\n    Deadline cooperative: завершение CUDA kernel и загрузка весов не прерываются.\n    Поэтому ≤180s подтверждается полным cold T4 замером, а не самим guard.\n    """\n    started = time.monotonic() if started is None else started\n    budget = float(config.get("budget_seconds", 180.0))\n    if not math.isfinite(budget) or not 0 < budget <= 180:\n        raise ValueError("Require 0 < budget_seconds <= 180")\n    policy = default_config()\n    guards_config = config.get("guards", {})\n    options = Options(**config.get("decoder", policy["decoder"]))\n    work_deadline = started + guards_config.get(\n        "generation", budget - min(20.0, budget * 0.12)\n    )\n    execution_deadline = started + guards_config.get(\n        "execution", budget - min(14.0, budget * 0.08)\n    )\n    verification_deadline = started + guards_config.get(\n        "verification", budget - min(10.0, budget * 0.06)\n    )\n    greedy_deadline = started + guards_config.get(\n        "greedy", budget - min(50.0, budget * 0.28)\n    )\n    specs = config["models"]\n    if not 2 <= len(specs) <= 4 or len({m["name"] for m in specs}) != len(specs):\n        raise ValueError("Require an ordered list of 2–4 distinct models")\n    if any(\n        not isinstance(m["K1"], int)\n        or not isinstance(m["K2"], int)\n        or not 1 <= m["K1"] <= m["K2"]\n        for m in specs\n    ):\n        raise ValueError("Require integer 1 <= K1 <= K2")\n    e2_primary = config.get("e2_primary", False)\n    if e2_primary:\n        if ([m["K1"] for m in specs] != [1, 1, 1] or\n            [m["K2"] for m in specs] != [2, 8, 8] or\n            [m.get("K3", m["K2"]) for m in specs] != [2, 16, 8] or\n            [bool(m.get("mixed_temperature")) for m in specs] != [False, True, True]):\n            raise ValueError("E2 primary requires C2/P8/W8 mixed, stage3 P16")\n    threshold = config.get("unanimity_threshold", 1.0)\n    if not 0 < threshold <= 1:\n        raise ValueError("Require 0 < unanimity_threshold <= 1")\n    device = torch.device(config.get("device", "cuda:0"))\n    configure_runtime()\n    if device.type == "cuda" and (\n        torch.cuda.device_count() != 1 or "T4" not in torch.cuda.get_device_name(0)\n    ):\n        raise ValueError("Official timing requires one NVIDIA T4")\n    if rows is None:\n        rows = json.loads(Path(config["rows"]).read_text())\n    if not rows or not all(isinstance(r["question"], str) for r in rows):\n        raise ValueError("Expected a JSON list with question strings")\n    model_hashes = {}\n    if models is None:\n        for spec in specs:\n            path = Path(config["weights_root"]) / spec["filename"]\n            model_hashes[spec["name"]] = sha256(path)\n            if model_hashes[spec["name"]] != spec["sha256"]:\n                raise ValueError("Weight SHA mismatch: " + spec["name"])\n        models = {\n            m["name"]: load_model(Path(config["weights_root"]) / m["filename"], device)\n            for m in specs\n        }\n    if tok is None:\n        if sha256(config["tokenizer"]) != config["tokenizer_sha256"]:\n            raise ValueError("Tokenizer SHA mismatch")\n        tok, table = tokenizer(config["tokenizer"])\n    encoded = [encode_question(tok, row["question"]) for row in rows]\n    # Same nonstochastic warmup as v7; model RNG is reset below.\n    for model in models.values():\n        decode(\n            Context(model, torch.tensor([encoded[0]], device=device), 16),\n            options=options,\n        )\n    if device.type == "cuda":\n        torch.cuda.synchronize(device)\n    records = [dict(greedy=None, samples={m["name"]: [] for m in specs}) for _ in rows]\n    lanes = config.get("sequence_budget", policy["sequence_budget"])\n    greedy_batch = question_batch(config.get("greedy_sequence_budget", lanes), 1)\n    extras = {m["name"]: m["K2"] - m["K1"] for m in specs}\n    wave_size = (\n        question_batch(lanes, max(extras.values())) if any(extras.values()) else 0\n    )\n    cap = config.get("cap", 512)\n    rng, effective_batches = {}, {}\n    guards, batches, oom_attempts = [], [], []\n    executor = Executor(config.get("generation_workers", policy["generation_workers"]))\n\n    def generate(name, order, batch, k, temperature, target, deadline):\n        batch = min(batch, effective_batches.get((name, k), batch))\n        offset = 0\n        while offset < len(order):\n            check_deadline(deadline)\n            selected = order[offset : offset + batch]\n            cpu_rng = torch.get_rng_state() if temperature is not None else None\n            gpu_rng = (\n                torch.cuda.get_rng_state(device)\n                if temperature is not None and device.type == "cuda"\n                else None\n            )\n            src = context = ids = lps = None\n            oom = False\n            try:\n                src = pad_sequence(\n                    [torch.tensor(encoded[i]) for i in selected],\n                    batch_first=True,\n                    padding_value=PAD,\n                ).to(device)\n                context = Context(models[name], src, cap)\n                ids, lps, _ = decode(\n                    context,\n                    k,\n                    temperature,\n                    deadline,\n                    compute_logprob=temperature is not None,\n                    options=options,\n                )\n                groups = pack(ids, lps, table, greedy=temperature is None)\n                for i, group in zip(selected, groups):\n                    if target == "greedy":\n                        records[i]["greedy"] = group[0]\n                    else:\n                        records[i]["samples"][target].extend(group)\n                executor.submit([c for group in groups for c in group])\n                batches.append(dict(model=name, target=target, ids=selected, k=k))\n                offset += len(selected)\n            except torch.cuda.OutOfMemoryError:\n                oom = True\n                oom_attempts.append(dict(model=name, offset=offset, batch=batch, k=k))\n                if cpu_rng is not None:\n                    torch.set_rng_state(cpu_rng)\n                if gpu_rng is not None:\n                    torch.cuda.set_rng_state(gpu_rng, device)\n                batch //= 2\n                if batch < 1:\n                    raise\n                effective_batches[name, k] = batch\n            finally:\n                del src, context, ids, lps\n                if oom:\n                    import gc\n\n                    gc.collect()\n                if device.type == "cuda" and (oom or config.get("release_cache_between_batches", False)):\n                    torch.cuda.empty_cache()\n\n    stopped, escalated, third_rows = [], [], []\n    try:\n        order = sorted(range(len(rows)), key=lambda i: (len(encoded[i]), i))\n        generate(\n            specs[0]["name"], order, greedy_batch, 1, None, "greedy", greedy_deadline\n        )\n        output = Path(config.get("output", "."))\n        output.mkdir(parents=True, exist_ok=True)\n        write_answers(\n            output / "greedy-fallback.jsonl", [r["greedy"]["code"] for r in records]\n        )\n        (output / "greedy-ready.json").write_text(json.dumps(dict(rows=len(rows))))\n        # An unfinished greedy pass is an error: never invent a missing answer.\n        for spec in specs:\n            name = spec["name"]\n            torch.manual_seed(config.get("seed", policy["seed"]))\n            if device.type == "cuda":\n                torch.cuda.manual_seed_all(config.get("seed", policy["seed"]))\n            try:\n                generate(\n                    name,\n                    order,\n                    question_batch(lanes, spec["K1"]),\n                    spec["K1"],\n                    config.get("temperature", policy["temperature"]),\n                    name,\n                    work_deadline,\n                )\n            except BudgetExpired:\n                guards.append("stage1_" + name)\n            rng[name] = (\n                torch.get_rng_state(),\n                torch.cuda.get_rng_state(device) if device.type == "cuda" else None,\n            )\n            if guards:\n                break\n        try:\n            if not guards:\n                if not executor.drain(work_deadline):\n                    raise BudgetExpired("Stage-one execution incomplete")\n                leaders = {}\n                for i in order:\n                    check_deadline(work_deadline)\n                    executor.attach(candidates(records[i], specs, "K1"))\n                    decision = stage1_decision(records[i], specs, threshold)\n                    if decision is None:\n                        raise BudgetExpired("Stage-one execution incomplete")\n                    if decision:\n                        stopped.append(i)\n                    else:\n                        escalated.append(i)\n                        leaders[i] = stage1_leader_fraction(records[i], specs)\n                priority = sorted(\n                    escalated, key=lambda i: (leaders[i], len(encoded[i]), i)\n                )\n                for offset in range(0, len(priority), wave_size) if wave_size else []:\n                    check_deadline(work_deadline)\n                    ready = priority[offset : offset + wave_size]\n                    for spec in specs:\n                        name = spec["name"]\n                        if not extras[name]:\n                            continue\n                        cpu_rng, gpu_rng = rng[name]\n                        torch.set_rng_state(cpu_rng)\n                        if gpu_rng is not None:\n                            torch.cuda.set_rng_state(gpu_rng, device)\n                        generate(\n                            name,\n                            ready,\n                            wave_size,\n                            extras[name],\n                            slot_temperatures(spec, spec["K1"] + 1, spec["K2"], config.get("temperature", policy["temperature"]))\n                            if e2_primary else config.get("temperature", policy["temperature"]),\n                            name,\n                            work_deadline,\n                        )\n                        rng[name] = (\n                            torch.get_rng_state(),\n                            (\n                                torch.cuda.get_rng_state(device)\n                                if device.type == "cuda"\n                                else None\n                            ),\n                        )\n                if e2_primary:\n                    if not executor.drain(work_deadline):\n                        raise BudgetExpired("Stage-two execution incomplete")\n                    executor.attach([c for i in escalated for c in candidates(records[i], specs)])\n                    # Decide only after all 19 stage-two votes have executed.\n                    third_rows = [i for i in priority if winner_fraction(records[i], specs) < .5]\n                    spec = specs[1]\n                    name = spec["name"]\n                    k = spec["K3"] - spec["K2"]\n                    batch = question_batch(lanes, k)\n                    cpu_rng, gpu_rng = rng[name]\n                    torch.set_rng_state(cpu_rng)\n                    if gpu_rng is not None:\n                        torch.cuda.set_rng_state(gpu_rng, device)\n                    generate(name, third_rows, batch, k,\n                             slot_temperatures(spec, spec["K2"] + 1, spec["K3"], config.get("temperature", policy["temperature"])),\n                             name, work_deadline)\n        except BudgetExpired:\n            guards.append("stage2_or_3" if e2_primary else "stage2")\n        executor.resize(config.get("tail_workers", 4))\n        if not executor.drain(execution_deadline):\n            guards.append("execution")\n        executor.attach([c for record in records for c in candidates(record, specs)])\n    finally:\n        executor.close()\n    selected = [select_available(record, specs) for record in records]\n    winners = [item[0] for item in selected]\n    counts = [item[1] for item in selected]\n    fallback = [i for i, count in enumerate(counts) if count == 0]\n    stopped_set, third_set = set(stopped), set(third_rows)\n    partial = [\n        i\n        for i, count in enumerate(counts)\n        if 0 < count < sum(m["K1"] if i in stopped_set else m.get("K3", m["K2"]) if i in third_set else m["K2"] for m in specs)\n    ]\n    verification = verify_winners(\n        winners, verification_deadline, config.get("tail_workers", 4)\n    )\n    for i in verification["mismatch_rows"]:\n        winners[i] = records[i]["greedy"]\n    output = Path(config.get("output", "."))\n    output.mkdir(parents=True, exist_ok=True)\n    path = output / "answers.jsonl"\n    write_answers(path, [w["code"] for w in winners])\n    elapsed = time.monotonic() - started\n    result = dict(\n        path=str(path),\n        runtime=dict(torch=torch.__version__, cuda=torch.version.cuda,\n                     device=torch.cuda.get_device_name(device) if device.type == "cuda" else "cpu"),\n        elapsed_seconds=elapsed,\n        within_budget=elapsed <= budget,\n        rows=len(rows),\n        stopped=stopped,\n        escalated=escalated,\n        third_rows=third_rows,\n        e2_primary=e2_primary,\n        guards=guards,\n        verification=verification,\n        batches=batches,\n        fallback_rows=fallback,\n        partial_rows=partial,\n        oom_attempts=oom_attempts,\n        full_pipeline_seconds=elapsed,\n        model_sha256=model_hashes,\n        model_order=[m["name"] for m in specs],\n        sequence_budget=lanes,\n        answers_sha256=sha256(path),\n        question_order_sha256=hashlib.sha256(\n            json.dumps([r["question"] for r in rows], ensure_ascii=False).encode()\n        ).hexdigest(),\n        mean_sample_count=sum(len(v) for r in records for v in r["samples"].values())\n        / len(rows),\n        peak_reserved_bytes=(\n            torch.cuda.max_memory_reserved(device) if device.type == "cuda" else None\n        ),\n        peak_allocated_bytes=(\n            torch.cuda.max_memory_allocated(device) if device.type == "cuda" else None\n        ),\n        total_device_memory_bytes=(\n            torch.cuda.get_device_properties(device).total_memory\n            if device.type == "cuda"\n            else None\n        ),\n        no_numeric_winner_rows=[\n            i for i, w in enumerate(winners) if w["execution"]["status"] != "numeric"\n        ],\n    )\n    # Diagnostics and reference scoring happen after answers file close.\n    (output / "timing.json").write_text(\n        json.dumps(result, ensure_ascii=False, indent=2) + "\\n"\n    )\n    if config.get("reference"):\n        refs = json.loads(Path(config["reference"]).read_text())\n        matches = [\n            i\n            for i, (ref, w) in enumerate(zip(refs, winners))\n            if ref["status"] == w["execution"]["status"] == "numeric"\n            and math.isclose(\n                ref["value"], w["execution"]["value"], rel_tol=1e-6, abs_tol=1e-6\n            )\n        ]\n        assert len(refs) == len(winners)\n        quality = dict(\n            matches=len(matches),\n            n=len(rows),\n            accuracy_pp=100 * len(matches) / len(rows),\n            match_indices=matches,\n        )\n        (output / "quality.json").write_text(json.dumps(quality, indent=2) + "\\n")\n        with gzip.open(\n            output / "candidate-journal.json.gz", "wt", encoding="utf8"\n        ) as stream:\n            json.dump(records, stream, ensure_ascii=False)\n    return result\n', 'runner.py': '"""Torch-free supervisor для одного cold pipeline, без Kaggle API и credentials."""\n\nimport json\nimport os\nimport signal\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\n\nclass DeviceMemory:\n    """NVML device-wide used memory, sampled every supervisor poll (20 ms).\n\n    Includes CUDA context and allocator cache. Torch peak allocated/reserved\n    are recorded separately by the worker and are exact allocator counters.\n    """\n\n    def __init__(self, enabled):\n        self.peak = None\n        self.lib = self.handle = None\n        if not enabled:\n            return\n        try:\n            import ctypes\n\n            class Memory(ctypes.Structure):\n                _fields_ = [(n, ctypes.c_ulonglong) for n in ("total", "free", "used")]\n\n            lib = ctypes.CDLL("libnvidia-ml.so.1")\n            assert lib.nvmlInit_v2() == 0\n            handle = ctypes.c_void_p()\n            assert lib.nvmlDeviceGetHandleByIndex_v2(0, ctypes.byref(handle)) == 0\n            self.lib, self.handle, self.memory = lib, handle, Memory()\n            self.ctypes = ctypes\n        except (OSError, AssertionError):\n            return\n\n    def sample(self):\n        if (\n            self.lib is not None\n            and self.lib.nvmlDeviceGetMemoryInfo(\n                self.handle, self.ctypes.byref(self.memory)\n            )\n            == 0\n        ):\n            self.peak = max(self.peak or 0, self.memory.used)\n\n\ndef run_cold(config):\n    """Запустить свежий процесс, сохранить timer и готовый greedy при hard stop.\n\n    Нельзя повторно использовать каталог предыдущего запуска. Timer начинается\n    до создания процесса и включает imports/load/warmup/decode/exec/vote/write.\n    Диагностика и оценка tune идут после закрытия answers.jsonl.\n    """\n    output = Path(config["output"])\n    output.mkdir(parents=True, exist_ok=True)\n    if any(\n        (output / n).exists() for n in ["timing.json", "answers.jsonl", "worker.log"]\n    ):\n        raise ValueError("Cold run requires a fresh output directory")\n    config_path = output / "config.json"\n    config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2) + "\\n")\n    start = time.monotonic()\n    env = dict(\n        os.environ,\n        CUDA_VISIBLE_DEVICES="0",\n        HW2_PROCESS_START=str(start),\n        PYTHONHASHSEED="0",\n        TOKENIZERS_PARALLELISM="false",\n        OMP_NUM_THREADS="1",\n        MKL_NUM_THREADS="1",\n        OPENBLAS_NUM_THREADS="1",\n        CUBLAS_WORKSPACE_CONFIG=":4096:8",\n        PYTORCH_ALLOC_CONF=config["cuda_allocator"],\n    )\n    env.pop("PYTORCH_CUDA_ALLOC_CONF", None)\n    env["PYTHONPATH"] = (\n        str(Path(__file__).resolve().parent.parent)\n        + os.pathsep\n        + env.get("PYTHONPATH", "")\n    )\n    hard_stop = config.get("guards", {}).get("supervisor", 172.0)\n    stopped = False\n    memory = DeviceMemory(not config.get("dry_run", False))\n    with (output / "worker.log").open("w") as log:\n        child = subprocess.Popen(\n            [sys.executable, "-m", "hw2_codegen.worker", str(config_path.resolve())],\n            env=env,\n            stdout=log,\n            stderr=log,\n            start_new_session=True,\n        )\n        try:\n            while child.poll() is None:\n                memory.sample()\n                if time.monotonic() - start >= hard_stop:\n                    stopped = True\n                    break\n                time.sleep(0.02)\n        finally:\n            try:\n                os.killpg(child.pid, signal.SIGKILL)\n            except ProcessLookupError:\n                pass\n            child.wait(timeout=2.0)\n    timer = output / "timing.json"\n    if child.returncode or stopped or not timer.is_file():\n        if not (output / "greedy-ready.json").is_file():\n            raise RuntimeError(\n                "Cold worker failed before greedy completion; inspect worker.log"\n            )\n        (output / "answers.jsonl").write_bytes(\n            (output / "greedy-fallback.jsonl").read_bytes()\n        )\n        report = dict(\n            elapsed_seconds=time.monotonic() - start,\n            guards=["supervisor"],\n            rows=json.loads((output / "greedy-ready.json").read_text())["rows"],\n            path=str(output / "answers.jsonl"),\n            hard_stop=stopped,\n            exit_code=child.returncode,\n        )\n        report["full_pipeline_seconds"] = report["elapsed_seconds"]\n        timer.write_text(json.dumps(report, indent=2) + "\\n")\n    else:\n        report = json.loads(timer.read_text())\n    report["peak_device_used_bytes"] = memory.peak\n    report["device_memory_sampling_seconds"] = 0.02\n    report["status"] = "guarded" if report["guards"] else "complete"\n    report["within_budget"] = report["elapsed_seconds"] <= config.get(\n        "budget_seconds", 180.0\n    )\n    (output / "summary.json").write_text(json.dumps(report, indent=2) + "\\n")\n    return report\n', 'tokenizer_repr.py': '"""Byte-level BPE: PAD/BOS/EOS=0/1/2, BPE id+3, три compact mode tokens."""\n\nimport ast\nimport io\nimport tokenize\n\nfrom tokenizers import AddedToken, Tokenizer\n\nfrom .model import BOS, EOS, PAD, TOKEN_OFFSET\n\nMODES = {"header_tail": "<|header_tail|>", "header": "<|header|>", "raw": "<|raw|>"}\n\n\ndef no_comments(code):\n    """Удалить только комментарии, сохранив строки."""\n    lines = code.splitlines(keepends=True)\n    try:\n        spans = [\n            t\n            for t in tokenize.generate_tokens(io.StringIO(code).readline)\n            if t.type == tokenize.COMMENT\n        ]\n    except (tokenize.TokenError, IndentationError, SyntaxError):\n        return code\n    for t in reversed(spans):\n        i = t.start[0] - 1\n        lines[i] = lines[i][: t.start[1]].rstrip(" \\t") + lines[i][t.end[1] :]\n    return "".join(lines)\n\n\ndef no_blank(code):\n    """Убрать пустые строки только при неизменном AST."""\n    protected = set()\n    try:\n        for t in tokenize.generate_tokens(io.StringIO(code).readline):\n            if t.type == tokenize.STRING:\n                protected.update(range(t.start[0], t.end[0] + 1))\n        candidate = "".join(\n            (\n                line\n                for i, line in enumerate(code.splitlines(keepends=True), 1)\n                if line.strip() or i in protected\n            )\n        )\n        if ast.dump(ast.parse(code)) == ast.dump(ast.parse(candidate)):\n            return candidate\n    except (tokenize.TokenError, SyntaxError, ValueError, RecursionError):\n        pass\n    return code\n\n\ndef restore_compact(text):\n    """Восстановить полный solve() по явному токену режима."""\n    mode = next((k for k, token in MODES.items() if text.startswith(token)), None)\n    if mode is None:\n        raise ValueError("Missing representation mode")\n    fragment = text[len(MODES[mode]) :]\n    if mode == "raw":\n        return fragment\n    if mode == "header":\n        return "def solve() -> int:\\n" + fragment\n    lines = fragment.splitlines(keepends=True)\n    if not lines or not lines[-1].startswith("    "):\n        raise ValueError("Missing final result expression")\n    lines[-1] = "    result = " + lines[-1][4:]\n    body = "".join(lines)\n    return (\n        "def solve() -> int:\\n"\n        + body\n        + ("" if body.endswith("\\n") else "\\n")\n        + "    return result\\n"\n    )\n\n\ndef compact(code):\n    """Удалить шаблон solve/result/return, проверив равенство AST."""\n    normalized = no_blank(no_comments(code))\n    raw = MODES["raw"] + normalized\n    lines = normalized.splitlines(keepends=True)\n    if not lines or lines[0] != "def solve() -> int:\\n":\n        return raw\n    try:\n        tree = ast.parse(normalized)\n        f = tree.body[0]\n        if len(tree.body) != 1 or not isinstance(f, ast.FunctionDef):\n            return raw\n        tail = (\n            len(f.body) >= 2\n            and isinstance(f.body[-1], ast.Return)\n            and isinstance(f.body[-1].value, ast.Name)\n            and (f.body[-1].value.id == "result")\n            and isinstance(f.body[-2], ast.Assign)\n            and (len(f.body[-2].targets) == 1)\n            and isinstance(f.body[-2].targets[0], ast.Name)\n            and (f.body[-2].targets[0].id == "result")\n            and (lines[-1].strip() == "return result")\n            and lines[-2].startswith("    result = ")\n        )\n        if tail:\n            lines[-2] = "    " + lines[-2][len("    result = ") :]\n            text = MODES["header_tail"] + "".join(lines[1:-1])\n        else:\n            text = MODES["header"] + "".join(lines[1:])\n        return (\n            text\n            if ast.dump(ast.parse(restore_compact(text))) == ast.dump(tree)\n            else raw\n        )\n    except (SyntaxError, ValueError, IndexError, RecursionError):\n        return raw\n\n\ndef encode(tokenizer, text):\n    return [\n        i + TOKEN_OFFSET for i in tokenizer.encode(text, add_special_tokens=False).ids\n    ]\n\n\ndef decode_bytes(tokenbytes, ids):\n    chunks = []\n    for i in ids:\n        if i == EOS:\n            break\n        if i >= TOKEN_OFFSET:\n            chunks.append(tokenbytes[i])\n    return b"".join(chunks)\n\n\ndef active_tokens(ids):\n    ids = [int(i) for i in ids if i != PAD]\n    return ids[: ids.index(EOS) + 1] if EOS in ids else ids\n\n\ndef tokenizer(path, variant=30):\n    if variant != 30:\n        raise ValueError("Submission uses compact representation 30")\n    tok = Tokenizer.from_file(str(path))\n    assert tok.get_vocab_size() == 8189\n    added = list(MODES.values()) if variant == 30 else []\n    assert tok.add_special_tokens(\n        [AddedToken(t, normalized=False, special=True) for t in added]\n    ) == len(added)\n    visible = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))\n    bchars = {b: chr(b) for b in visible}\n    for i, b in enumerate(b for b in range(256) if b not in bchars):\n        bchars[b] = chr(256 + i)\n    charbytes = {c: b for b, c in bchars.items()}\n    tokenbytes = [b"", b"", b""] + [\n        bytes(charbytes[c] for c in tok.id_to_token(i)) for i in range(8189)\n    ]\n    tokenbytes += [t.encode() for t in added]\n    assert all(\n        (\n            tok.encode(t, add_special_tokens=False).ids == [8189 + i]\n            for i, t in enumerate(added)\n        )\n    )\n    return (tok, tokenbytes)\n\n\ndef encode_question(tok, question):\n    """Вопрос без переписывания чисел; BOS+content+EOS."""\n    return [BOS] + encode(tok, question) + [EOS]\n', 'train.py': '"""WSD-рецепт L1–L3/P1 без Kaggle orchestration.\n\nBatch 128, AdamW wd=.01, clip=1, AMP FP16/GradScaler1024; constant main,\nзатем disposable cooldown. Main и cooldown разделены checkpoint-ом.\nСкрипт не скачивает данные и не запускает обучение при импорте.\n"""\n\nimport argparse\nimport copy\nimport json\nimport math\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\n\nfrom .data import CompactCache, collate, make_order, take\nfrom .model import PAD, from_spec\n\nMODEL_C = dict(\n    d_model=512,\n    heads=8,\n    encoder_layers=6,\n    decoder_layers=6,\n    swiglu_dim=1368,\n    vocab=8195,\n)\nCOMPACT_EDGES = (53, 79, 105)\n\n\ndef seed_all(seed=42):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef learning_rate(update, peak, warmup, main_updates, cooldown_updates, kind="sqrt"):\n    """WSD на оси updates: warmup → constant → linear или 1-sqrt до нуля.\n\n    Боевые L2/L3 задавали длительность main по wall time; точное число updates\n    и старт cooldown берутся из training.json выбранного checkpoint.\n    """\n    if update <= main_updates:\n        return peak * min(1.0, update / max(1, warmup))\n    position = min(1.0, (update - main_updates) / max(1, cooldown_updates))\n    if kind not in ("sqrt", "linear"):\n        raise ValueError("cooldown must be sqrt or linear")\n    return peak * (1.0 - (math.sqrt(position) if kind == "sqrt" else position))\n\n\ndef truncation_mask(scores, lengths, q=0.1, edges=COMPACT_EDGES):\n    """Глобальная mask effective batch: удалить q самых трудных, с квотами по длине.\n\n    Округление q*N, largest remainder, минимум один retained пример в каждом bin.\n    При равной CE побеждает исходная позиция. Это точная политика P1.\n    """\n    scores = np.asarray(scores, dtype=np.float64)\n    lengths = np.asarray(lengths, dtype=np.int64)\n    if (\n        scores.ndim != 1\n        or scores.shape != lengths.shape\n        or not np.isfinite(scores).all()\n        or not 0 <= q < 1\n    ):\n        raise FloatingPointError("Invalid per-example loss")\n    bins = np.searchsorted(edges, lengths, side="right")\n    counts = np.bincount(bins, minlength=len(edges) + 1)\n    capacities = np.maximum(0, counts - 1)\n    desired = min(int(math.floor(q * len(scores) + 0.5)), int(capacities.sum()))\n    quotas = np.minimum(np.floor(q * counts).astype(int), capacities)\n    while int(quotas.sum()) < desired:\n        available = [b for b in range(len(counts)) if quotas[b] < capacities[b]]\n        b = min(available, key=lambda b: (-(q * counts[b] - quotas[b]), b))\n        quotas[b] += 1\n    keep = np.ones(len(scores), dtype=bool)\n    for b, quota in enumerate(quotas):\n        positions = np.flatnonzero(bins == b)\n        ranked = sorted(positions, key=lambda j: (-scores[j], int(j)))\n        keep[ranked[:quota]] = False\n    return keep\n\n\ndef update(model, optimizer, scaler, cache, indices, device, micro, lr, q=0.0):\n    """Один effective batch; token-weighted CE, EOS учитывается, PAD игнорируется.\n\n    q=0 повторяет L1.update. q>0 повторяет P1 graph_truncated_update: mask по\n    всему effective batch до backward, знаменатель — число retained tokens.\n    При OOM/нечисловом состоянии останавливаемся: безопасно продолжать из full\n    checkpoint. Автоматические откаты и выбор micro остаются в work/.\n    """\n    optimizer.zero_grad(set_to_none=True)\n    total = 0.0\n    lengths = np.asarray([len(cache[int(i)][1]) for i in indices])\n    tokens = int(sum(lengths + 1))\n    graphs, scores = [], []\n    for lo in range(0, len(indices), micro):\n        src, tgt, labels, _ = collate(cache, indices[lo : lo + micro], device)\n        with torch.autocast(\n            device_type=device.type, dtype=torch.float16, enabled=scaler.is_enabled()\n        ):\n            logits = model(src, tgt)\n            if q:\n                losses = (\n                    F.cross_entropy(\n                        logits.reshape(-1, model.vocab),\n                        labels.reshape(-1),\n                        ignore_index=PAD,\n                        reduction="none",\n                    )\n                    .reshape_as(labels)\n                    .sum(1)\n                )\n            else:\n                loss = (\n                    F.cross_entropy(\n                        logits.reshape(-1, model.vocab),\n                        labels.reshape(-1),\n                        ignore_index=PAD,\n                        reduction="sum",\n                    )\n                    / tokens\n                )\n        if q:\n            values = losses.detach() / labels.ne(PAD).sum(1)\n            if not bool(torch.isfinite(values).all()):\n                raise FloatingPointError("Nonfinite selection loss")\n            scores.extend(values.cpu().tolist())\n            graphs.append(losses)\n            del losses, values\n        else:\n            if not bool(torch.isfinite(loss).detach()):\n                raise FloatingPointError("Nonfinite loss")\n            scaler.scale(loss).backward()\n            total += float(loss.detach())\n            del loss\n        del src, tgt, labels, logits\n    keep = np.ones(len(indices), dtype=bool)\n    if q:\n        keep = truncation_mask(scores, lengths, q)\n        tokens = int(sum(lengths[keep] + 1))\n        lo = 0\n        while graphs:\n            losses = graphs.pop(0)\n            mask = torch.as_tensor(keep[lo : lo + len(losses)], device=device)\n            loss = losses[mask].sum() / tokens\n            if not bool(torch.isfinite(loss).detach()):\n                raise FloatingPointError("Nonfinite retained loss")\n            scaler.scale(loss).backward()\n            total += float(loss.detach())\n            lo += len(losses)\n            del losses, loss, mask\n    for group in optimizer.param_groups:\n        group["lr"] = lr\n    scaler.unscale_(optimizer)\n    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n    skipped = (\n        bool(\n            any(float(t) > 0 for t in scaler._found_inf_per_device(optimizer).values())\n        )\n        if scaler.is_enabled()\n        else False\n    )\n    if not skipped and not bool(torch.isfinite(norm)):\n        raise FloatingPointError("Nonfinite successful gradient")\n    scaler.step(optimizer)\n    scaler.update()\n    tensors = list(model.parameters()) + [\n        value\n        for state in optimizer.state.values()\n        for value in state.values()\n        if torch.is_tensor(value) and value.is_floating_point()\n    ]\n    if not bool(torch.isfinite(torch.stack(torch._foreach_norm(tensors))).all()):\n        raise FloatingPointError("Nonfinite model/Adam after update; restore checkpoint")\n    return dict(\n        CE=total,\n        tokens=tokens,\n        AMP_skip=int(skipped),\n        keep=keep.tolist(),\n        gradient_norm=float(norm) if torch.isfinite(norm) else None,\n    )\n\n\ndef save_checkpoint(path, model, optimizer, scaler, state, model_spec):\n    """Полное состояние перед cooldown; model-only экспортируется отдельно."""\n    torch.save(\n        dict(\n            model=model.state_dict(),\n            optimizer=optimizer.state_dict(),\n            scaler=scaler.state_dict(),\n            state=copy.deepcopy(state),\n            model_spec=model_spec,\n            torch_rng=torch.get_rng_state(),\n            python_rng=random.getstate(),\n            numpy_rng=np.random.get_state(),\n            cuda_rng=torch.cuda.get_rng_state_all()\n            if torch.cuda.is_available()\n            else [],\n        ),\n        path,\n    )\n\n\ndef train(config):\n    """Запустить явное число updates по сохранённому order.npy либо L1 permutation."""\n    device = torch.device(config.get("device", "cuda"))\n    seed_all(config.get("seed", 42))\n    cache = CompactCache(config["cache"])\n    order = (\n        np.load(config["order"], allow_pickle=False)\n        if config.get("order")\n        else make_order(cache)\n    )\n    spec = config.get("model_spec", MODEL_C)\n    model = from_spec(spec, dropout=config.get("dropout", 0.0)).to(device).train()\n    optimizer = torch.optim.AdamW(\n        model.parameters(),\n        lr=config["peak_lr"],\n        weight_decay=0.01,\n        **({"fused": True} if device.type == "cuda" else {}),\n    )\n    scaler = torch.amp.GradScaler(\n        device.type, enabled=device.type == "cuda", init_scale=1024\n    )\n    state = dict(updates=0, cursor=0, AMP_skips=0)\n    if config.get("resume"):\n        # Load only one\'s own trusted full training checkpoint (contains numpy RNG).\n        payload = torch.load(config["resume"], map_location=device, weights_only=False)\n        if payload["model_spec"] != spec:\n            raise ValueError("Resume model_spec mismatch")\n        model.load_state_dict(payload["model"])\n        optimizer.load_state_dict(payload["optimizer"])\n        scaler.load_state_dict(payload["scaler"])\n        state = payload["state"]\n        torch.set_rng_state(payload["torch_rng"].cpu())\n        random.setstate(payload["python_rng"])\n        np.random.set_state(payload["numpy_rng"])\n        if payload["cuda_rng"]:\n            torch.cuda.set_rng_state_all([s.cpu() for s in payload["cuda_rng"]])\n    output = Path(config["output"])\n    output.mkdir(parents=True, exist_ok=True)\n    main, cooldown = config["main_updates"], config["cooldown_updates"]\n    with (output / "training.jsonl").open("a") as log:\n        for step in range(state["updates"] + 1, main + cooldown + 1):\n            if step == main + 1:\n                save_checkpoint(\n                    output / "stable.pt", model, optimizer, scaler, state, spec\n                )\n            lr = learning_rate(\n                step,\n                config["peak_lr"],\n                config.get("warmup_updates", 1000),\n                main,\n                cooldown,\n                config.get("cooldown", "linear"),\n            )\n            q = (\n                config.get("cooldown_truncation", 0.0)\n                if step > main\n                else config.get("main_truncation", 0.0)\n            )\n            indices = take(order, state["cursor"], config.get("batch", 128))\n            result = update(\n                model,\n                optimizer,\n                scaler,\n                cache,\n                indices,\n                device,\n                config.get("micro_batch", 64),\n                lr,\n                q,\n            )\n            state.update(\n                updates=step,\n                cursor=state["cursor"] + len(indices),\n                AMP_skips=state["AMP_skips"] + result["AMP_skip"],\n            )\n            log.write(json.dumps(dict(step=step, lr=lr, q=q, **result)) + "\\n")\n            if step % config.get("checkpoint_every", 1000) == 0:\n                save_checkpoint(\n                    output / "latest.pt", model, optimizer, scaler, state, spec\n                )\n        save_checkpoint(output / "latest.pt", model, optimizer, scaler, state, spec)\n    torch.save(\n        dict(\n            format="nlp-hw02-model-only-v1",\n            optimizer_included=False,\n            model_spec=spec,\n            model={k: v.detach().cpu().half() for k, v in model.state_dict().items()},\n        ),\n        output / "weights-fp16.pt",\n    )\n    return state\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("config", type=Path)\n    args = parser.parse_args()\n    train(json.loads(args.config.read_text()))\n', 'vote.py': '"""Числовые ответы: граф isclose(1e-6), число голосов → greedy → mean LP → порядок.\n\nisclose нетранзитивен: кластер — связная компонента, а не округлённое значение.\nОдинаковые программы экономят исполнение, но каждая sample сохраняет свой голос.\n"""\n\nimport math\n\n\ndef numeric(candidate):\n    execution = candidate.get("execution", {})\n    value = execution.get("value")\n    return (\n        execution.get("status") == "numeric"\n        and isinstance(value, (int, float))\n        and math.isfinite(value)\n    )\n\n\ndef same(a, b):\n    return math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-6)\n\n\ndef select(greedy, main, long, k=8):\n    """Вернуть сам объект кандидата; greedy представляет свою winning component."""\n    samples = [\n        c\n        for c in main[:k] + long[:k]\n        if c.get("mean_logprob") is not None and math.isfinite(c["mean_logprob"])\n    ]\n    good = [c for c in [greedy] + samples if numeric(c)]\n    if not good:\n        return greedy\n    parent = list(range(len(good)))\n\n    def root(i):\n        while parent[i] != i:\n            parent[i] = parent[parent[i]]\n            i = parent[i]\n        return i\n\n    for i in range(len(good)):\n        for j in range(i):\n            if same(good[i]["execution"]["value"], good[j]["execution"]["value"]):\n                parent[root(i)] = root(j)\n    groups = {}\n    for i, candidate in enumerate(good):\n        groups.setdefault(root(i), []).append(candidate)\n    winning = max(\n        groups.values(),\n        key=lambda group: (\n            len(group),\n            any(c is greedy for c in group),\n            sum(c["mean_logprob"] for c in group) / len(group),\n        ),\n    )\n    if any(c is greedy for c in winning):\n        return greedy\n    return max(winning, key=lambda c: c["mean_logprob"])\n\n\ndef candidates(record, models, stage=None):\n    return [record["greedy"]] + [\n        c for m in models for c in record["samples"][m["name"]][: m.get(stage or "K3", m["K2"])]\n    ]\n\n\ndef stage1_decision(record, models, threshold=1.0):\n    if record["greedy"] is None or any(\n        len(record["samples"][m["name"]]) < m["K1"] for m in models\n    ):\n        return None\n    first = candidates(record, models, "K1")\n    if any(\n        c.get("execution", {}).get("status", "budget_not_evaluated")\n        == "budget_not_evaluated"\n        for c in first\n    ):\n        return None\n    if not all(numeric(c) for c in first):\n        return False\n    return (\n        sum(\n            same(c["execution"]["value"], first[0]["execution"]["value"]) for c in first\n        )\n        / len(first)\n        >= threshold\n    )\n\n\ndef stage1_leader_fraction(record, models):\n    if stage1_decision(record, models) is None:\n        raise ValueError("Priority requires completed stage-one executions")\n    first = candidates(record, models, "K1")\n    remaining = [c for c in first if numeric(c)]\n    largest = 0\n    while remaining:\n        component = [remaining.pop(0)]\n        while True:\n            adjacent = [\n                c\n                for c in remaining\n                if any(\n                    same(c["execution"]["value"], v["execution"]["value"])\n                    for v in component\n                )\n            ]\n            if not adjacent:\n                break\n            component.extend(adjacent)\n            identities = {id(c) for c in adjacent}\n            remaining = [c for c in remaining if id(c) not in identities]\n        largest = max(largest, len(component))\n    return largest / len(first)\n\n\ndef select_available(record, models, **kwargs):\n    available = [\n        c\n        for c in candidates(record, models)[1:]\n        if c.get("execution", {}).get("status", "budget_not_evaluated")\n        != "budget_not_evaluated"\n    ]\n    return (\n        select(record["greedy"], available, [], k=len(available), **kwargs),\n        len(available),\n    )\n\n\ndef replay_record(greedy, samples, models, threshold=1.0):\n    record = {\n        "greedy": greedy,\n        "samples": {m["name"]: samples[m["name"]][: m["K1"]] for m in models},\n    }\n    decision = stage1_decision(record, models, threshold)\n    if decision is None:\n        raise ValueError("Offline replay requires completed stage-one executions")\n    if not decision:\n        record["samples"] = {m["name"]: samples[m["name"]][: m["K2"]] for m in models}\n    winner, count = select_available(record, models)\n    return (winner, decision, count)\n\n\ndef winner_fraction(record, models):\n    """Largest voting numeric component / ALL stage-two votes, failures included."""\n    votes = candidates(record, models, "K2")\n    remaining = [c for i, c in enumerate(votes) if numeric(c) and\n                 (i == 0 or (c.get("mean_logprob") is not None and math.isfinite(c["mean_logprob"])))]\n    largest = 0\n    while remaining:\n        component = [remaining.pop(0)]\n        while True:\n            adjacent = [c for c in remaining if any(same(c["execution"]["value"], v["execution"]["value"]) for v in component)]\n            if not adjacent:\n                break\n            component.extend(adjacent)\n            identities = {id(c) for c in adjacent}\n            remaining = [c for c in remaining if id(c) not in identities]\n        largest = max(largest, len(component))\n    return largest / len(votes)\n\n\ndef slot_temperatures(spec, first, last, default=.7):\n    """Slots are one-based; E2 mixes every fourth P/W sample at T=1."""\n    return [1.0 if spec.get("mixed_temperature") and slot % 4 == 0 else default\n            for slot in range(first, last + 1)]\n\n\ndef replay_policy(greedy, samples, models, *, e2_primary=False, threshold=1.0):\n    """Production policy on a completed journal, including conditional E2 stage3."""\n    record = dict(greedy=greedy, samples={m["name"]: samples[m["name"]][:m["K1"]] for m in models})\n    stop = stage1_decision(record, models, threshold)\n    if stop is None:\n        raise ValueError("Offline replay requires completed stage-one executions")\n    third = False\n    if not stop:\n        record["samples"] = {m["name"]: samples[m["name"]][:m["K2"]] for m in models}\n        third = e2_primary and winner_fraction(record, models) < .5\n        if third:\n            record["samples"] = {m["name"]: samples[m["name"]][:m.get("K3", m["K2"])] for m in models}\n    winner, count = select_available(record, models)\n    return winner, stop, third, count\n', 'worker.py': '"""Cold worker: timestamp перед Torch import; один split на одном процессе/GPU."""\n\nimport os\nimport signal\nimport sys\nimport time\n\nSTARTED = float(os.environ.get("HW2_PROCESS_START", time.monotonic()))\nif sys.platform.startswith("linux"):\n    import ctypes\n\n    parent = os.getppid()\n    signal.signal(signal.SIGTERM, lambda *_: os.killpg(os.getpgrp(), signal.SIGKILL))\n    assert ctypes.CDLL(None).prctl(1, signal.SIGTERM, 0, 0, 0) == 0\n    if parent == 1 or os.getppid() != parent:\n        os.killpg(os.getpgrp(), signal.SIGKILL)\n\nimport json\nfrom pathlib import Path\nfrom .pipeline import run_inference\n\nif __name__ == "__main__":\n    config = json.loads(Path(sys.argv[1]).read_text())\n    fixture = {}\n    if config.get("dry_run"):\n        from .dry_run import fixtures\n\n        rows, models, tok, table, specs = fixtures()\n        if config.get("variant"):\n            # Exercise the exact variant policy with cheap CPU fixtures.\n            tiny = list(models.values())\n            models = {m["name"]: tiny[i % len(tiny)] for i, m in enumerate(config["models"])}\n        else:\n            config["models"] = specs\n        config.update(cap=32, sequence_budget=16, greedy_sequence_budget=16)\n        fixture = dict(rows=rows, models=models, tok=tok, table=table)\n    run_inference(config, started=STARTED, **fixture)\n'}
INPUT_FILES = json.loads(zlib.decompress(base64.b64decode('...')))
CONFIGS = [{'models': [{'name': 'W-B7', 'filename': 'w-b7-fp16.pt', 'sha256': '3378c368f601c7cd046d40de22cf5cc6f5081deff190819bb4f616f0eb2761ac', 'dataset': 'friend1/nlp-hw02-b7', 'source_kind': 'notebooks', 'source_ref': 'friend1/nlp-hw02-b7/1', 'K1': 1, 'K2': 1, 'source_path': 'model-GPU0/weights/b6-final-fp16.pt'}, {'name': 'C-long2x-B7', 'filename': 'c-long2x-b7-fp16.pt', 'sha256': 'bcb41c54d031b70523d357dd7d14dc24ec7d1393a6cb8714d8eacfe761cca32b', 'dataset': 'friend1/nlp-hw02-b7', 'source_kind': 'notebooks', 'source_ref': 'friend1/nlp-hw02-b7/1', 'K1': 1, 'K2': 8, 'source_path': 'model-GPU1/weights/b6-final-fp16.pt'}, {'name': 'W-B6', 'filename': 'w-b6-fp16.pt', 'sha256': 'cb34c81e6cda83c73bf4662c5c425446be8cdb48c30dd5e9e217fd6cbc1cd496', 'dataset': 'friend1/nlp-hw02-b6', 'source_kind': 'notebooks', 'source_ref': 'friend1/nlp-hw02-b6/1', 'K1': 1, 'K2': 8, 'source_path': 'model-GPU0/weights/b6-final-fp16.pt'}], 'unanimity_threshold': 1.0, 'sequence_budget': 640, 'greedy_sequence_budget': 640, 'scorer_enabled': False, 'scorer_weight': 0.0, 'device': 'cuda:0', 'budget_seconds': 180.0, 'cap': 512, 'seed': 270042, 'temperature': 0.7, 'tail_workers': 4, 'generation_workers': 2, 'expected_tune_matches': 1547, 'tokenizer_sha256': '261f25b5d4be006a6a00e41e0eafbe522580741da627ba670f6b9805a2659b7d', 'decoder': {'share_cross': True, 'async_eos': True, 'eos_every': 32, 'compact_done': True, 'compact_fraction': 0.25, 'race_sampler': True, 'lean_greedy': True}, 'guards': {'generation': 160.0, 'execution': 166.0, 'verification': 170.0, 'supervisor': 172.0, 'greedy': 130.0}, 'cuda_allocator': 'expandable_segments:True,garbage_collection_threshold:0.8', 'release_cache_between_batches': True, 'variant': 'v1'}, {'models': [{'name': 'W-B7', 'filename': 'w-b7-fp16.pt', 'sha256': '3378c368f601c7cd046d40de22cf5cc6f5081deff190819bb4f616f0eb2761ac', 'dataset': 'friend1/nlp-hw02-b7', 'source_kind': 'notebooks', 'source_ref': 'friend1/nlp-hw02-b7/1', 'K1': 1, 'K2': 1, 'source_path': 'model-GPU0/weights/b6-final-fp16.pt'}, {'name': 'C-long2x-B7', 'filename': 'c-long2x-b7-fp16.pt', 'sha256': 'bcb41c54d031b70523d357dd7d14dc24ec7d1393a6cb8714d8eacfe761cca32b', 'dataset': 'friend1/nlp-hw02-b7', 'source_kind': 'notebooks', 'source_ref': 'friend1/nlp-hw02-b7/1', 'K1': 1, 'K2': 8, 'source_path': 'model-GPU1/weights/b6-final-fp16.pt'}, {'name': 'pilot-b', 'filename': 'p1-b-fp16.pt', 'sha256': '8439037720dda16dda6b73fe676860d5498bbcba4cacebc0d709bbb22db464a5', 'dataset': 'friend1/nlp-hw02-kaggle-p1', 'source_kind': 'notebooks', 'source_ref': 'friend1/nlp-hw02-kaggle-p1/3', 'K1': 1, 'K2': 8, 'source_path': 'model-pilot-b/weights/long-truncation-fp16.pt'}], 'unanimity_threshold': 1.0, 'sequence_budget': 640, 'greedy_sequence_budget': 640, 'scorer_enabled': False, 'scorer_weight': 0.0, 'device': 'cuda:0', 'budget_seconds': 180.0, 'cap': 512, 'seed': 270042, 'temperature': 0.7, 'tail_workers': 4, 'generation_workers': 2, 'expected_tune_matches': 1547, 'tokenizer_sha256': '261f25b5d4be006a6a00e41e0eafbe522580741da627ba670f6b9805a2659b7d', 'decoder': {'share_cross': True, 'async_eos': True, 'eos_every': 32, 'compact_done': True, 'compact_fraction': 0.25, 'race_sampler': True, 'lean_greedy': True}, 'guards': {'generation': 160.0, 'execution': 166.0, 'verification': 170.0, 'supervisor': 172.0, 'greedy': 130.0}, 'cuda_allocator': 'expandable_segments:True,garbage_collection_threshold:0.8', 'release_cache_between_batches': True, 'variant': 'v2'}, {'models': [{'name': 'C-long2x-B6', 'filename': 'c-long2x-b6-fp16.pt', 'sha256': 'ee30794048bb6200cf97ac4d7e5c1606f0e97893ea04f6567c9487f6c04345fa', 'dataset': 'friend1/nlp-hw02-b6', 'source_kind': 'notebooks', 'source_ref': 'friend1/nlp-hw02-b6/1', 'K1': 1, 'K2': 1, 'source_path': 'model-GPU1/weights/b6-final-fp16.pt'}, {'name': 'pilot-b', 'filename': 'p1-b-fp16.pt', 'sha256': '8439037720dda16dda6b73fe676860d5498bbcba4cacebc0d709bbb22db464a5', 'dataset': 'friend1/nlp-hw02-kaggle-p1', 'source_kind': 'notebooks', 'source_ref': 'friend1/nlp-hw02-kaggle-p1/3', 'K1': 1, 'K2': 8, 'source_path': 'model-pilot-b/weights/long-truncation-fp16.pt'}, {'name': 'W-B7', 'filename': 'w-b7-fp16.pt', 'sha256': '3378c368f601c7cd046d40de22cf5cc6f5081deff190819bb4f616f0eb2761ac', 'dataset': 'friend1/nlp-hw02-b7', 'source_kind': 'notebooks', 'source_ref': 'friend1/nlp-hw02-b7/1', 'K1': 1, 'K2': 8, 'source_path': 'model-GPU0/weights/b6-final-fp16.pt'}], 'unanimity_threshold': 1.0, 'sequence_budget': 640, 'greedy_sequence_budget': 640, 'scorer_enabled': False, 'scorer_weight': 0.0, 'device': 'cuda:0', 'budget_seconds': 180.0, 'cap': 512, 'seed': 270042, 'temperature': 0.7, 'tail_workers': 4, 'generation_workers': 2, 'expected_tune_matches': 1547, 'tokenizer_sha256': '261f25b5d4be006a6a00e41e0eafbe522580741da627ba670f6b9805a2659b7d', 'decoder': {'share_cross': True, 'async_eos': True, 'eos_every': 32, 'compact_done': True, 'compact_fraction': 0.25, 'race_sampler': True, 'lean_greedy': True}, 'guards': {'generation': 160.0, 'execution': 166.0, 'verification': 170.0, 'supervisor': 172.0, 'greedy': 130.0}, 'cuda_allocator': 'expandable_segments:True,garbage_collection_threshold:0.8', 'release_cache_between_batches': True, 'variant': 'v3'}, {'models': [{'name': 'C-long2x-B6', 'filename': 'c-long2x-b6-fp16.pt', 'sha256': 'ee30794048bb6200cf97ac4d7e5c1606f0e97893ea04f6567c9487f6c04345fa', 'dataset': 'friend1/nlp-hw02-b6', 'source_kind': 'notebooks', 'source_ref': 'friend1/nlp-hw02-b6/1', 'K1': 1, 'K2': 2, 'source_path': 'model-GPU1/weights/b6-final-fp16.pt', 'K3': 2, 'mixed_temperature': False}, {'name': 'pilot-b', 'filename': 'p1-b-fp16.pt', 'sha256': '8439037720dda16dda6b73fe676860d5498bbcba4cacebc0d709bbb22db464a5', 'dataset': 'friend1/nlp-hw02-kaggle-p1', 'source_kind': 'notebooks', 'source_ref': 'friend1/nlp-hw02-kaggle-p1/3', 'K1': 1, 'K2': 8, 'source_path': 'model-pilot-b/weights/long-truncation-fp16.pt', 'K3': 16, 'mixed_temperature': True}, {'name': 'W-B6', 'filename': 'w-b6-fp16.pt', 'sha256': 'cb34c81e6cda83c73bf4662c5c425446be8cdb48c30dd5e9e217fd6cbc1cd496', 'dataset': 'friend1/nlp-hw02-b6', 'source_kind': 'notebooks', 'source_ref': 'friend1/nlp-hw02-b6/1', 'K1': 1, 'K2': 8, 'source_path': 'model-GPU0/weights/b6-final-fp16.pt', 'K3': 8, 'mixed_temperature': True}], 'unanimity_threshold': 1.0, 'sequence_budget': 640, 'greedy_sequence_budget': 640, 'scorer_enabled': False, 'scorer_weight': 0.0, 'device': 'cuda:0', 'budget_seconds': 180.0, 'cap': 512, 'seed': 270042, 'temperature': 0.7, 'tail_workers': 4, 'generation_workers': 2, 'expected_tune_matches': 1547, 'tokenizer_sha256': '261f25b5d4be006a6a00e41e0eafbe522580741da627ba670f6b9805a2659b7d', 'decoder': {'share_cross': True, 'async_eos': True, 'eos_every': 32, 'compact_done': True, 'compact_fraction': 0.25, 'race_sampler': True, 'lean_greedy': True}, 'guards': {'generation': 160.0, 'execution': 166.0, 'verification': 170.0, 'supervisor': 172.0, 'greedy': 130.0}, 'cuda_allocator': 'expandable_segments:True,garbage_collection_threshold:0.8', 'release_cache_between_batches': True, 'variant': 'v4', 'e2_primary': True}, {'models': [{'name': 'C-long2x-B6', 'filename': 'c-long2x-b6-fp16.pt', 'sha256': 'ee30794048bb6200cf97ac4d7e5c1606f0e97893ea04f6567c9487f6c04345fa', 'dataset': 'friend1/nlp-hw02-b6', 'source_kind': 'notebooks', 'source_ref': 'friend1/nlp-hw02-b6/1', 'K1': 1, 'K2': 1, 'source_path': 'model-GPU1/weights/b6-final-fp16.pt'}, {'name': 'W-B6', 'filename': 'w-b6-fp16.pt', 'sha256': 'cb34c81e6cda83c73bf4662c5c425446be8cdb48c30dd5e9e217fd6cbc1cd496', 'dataset': 'friend1/nlp-hw02-b6', 'source_kind': 'notebooks', 'source_ref': 'friend1/nlp-hw02-b6/1', 'K1': 1, 'K2': 8, 'source_path': 'model-GPU0/weights/b6-final-fp16.pt'}, {'name': 'C-long2x-B7', 'filename': 'c-long2x-b7-fp16.pt', 'sha256': 'bcb41c54d031b70523d357dd7d14dc24ec7d1393a6cb8714d8eacfe761cca32b', 'dataset': 'friend1/nlp-hw02-b7', 'source_kind': 'notebooks', 'source_ref': 'friend1/nlp-hw02-b7/1', 'K1': 1, 'K2': 8, 'source_path': 'model-GPU1/weights/b6-final-fp16.pt'}], 'unanimity_threshold': 1.0, 'sequence_budget': 640, 'greedy_sequence_budget': 640, 'scorer_enabled': False, 'scorer_weight': 0.0, 'device': 'cuda:0', 'budget_seconds': 180.0, 'cap': 512, 'seed': 270042, 'temperature': 0.7, 'tail_workers': 4, 'generation_workers': 2, 'expected_tune_matches': 1547, 'tokenizer_sha256': '261f25b5d4be006a6a00e41e0eafbe522580741da627ba670f6b9805a2659b7d', 'decoder': {'share_cross': True, 'async_eos': True, 'eos_every': 32, 'compact_done': True, 'compact_fraction': 0.25, 'race_sampler': True, 'lean_greedy': True}, 'guards': {'generation': 160.0, 'execution': 166.0, 'verification': 170.0, 'supervisor': 172.0, 'greedy': 130.0}, 'cuda_allocator': 'expandable_segments:True,garbage_collection_threshold:0.8', 'release_cache_between_batches': True, 'variant': 'v5'}]
for name, content in CODE_FILES.items():
    (CODE / name).write_text(content)
for name, content in INPUT_FILES.items():
    (INPUTS / name).write_text(content)
sys.path.insert(0, str(CODE.parent))
from hw2_codegen.runner import run_cold

def digest(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(4 * 1024**2), b""):
            h.update(block)
    return h.hexdigest()

# Mount resolution occurs before the individual timers. Weight SHA/load,
# tokenizer, imports, warmup, inference, execution/vote and write are timed
# separately from a fresh process for EACH variant/split.
if not DRY:
    weights = Path("/tmp/hw2-variants-weights")
    weights.mkdir(parents=True, exist_ok=True)
    for spec in {m["name"]:m for c in CONFIGS for m in c["models"]}.values():
        slug = spec["source_ref"].split("/")[1]
        mounts = [p for p in Path("/kaggle/input").rglob(slug) if p.is_dir()]
        assert len(mounts) == 1, (slug, len(mounts))
        target = mounts[0] / spec["source_path"]
        assert target.is_file(), str(target)
        link = weights / spec["filename"]
        if not link.exists():
            link.symlink_to(target)
        assert link.resolve() == target.resolve()
else:
    (INPUTS / "dry-reference.json").write_text(json.dumps([dict(status="numeric",value=1)]*8))
print(json.dumps(dict(status="ready", variants=len(CONFIGS), dry_run=DRY)), flush=True)


In [ ]:
SUMMARY = dict(training=False, cold_process_per_split=True, single_visible_gpu=True, variants={})
for template in CONFIGS:
    name = template["variant"]
    reports = {}
    for split in ("test", "tune"):
        cfg = json.loads(json.dumps(template))
        out = ROOT / "variants" / name
        if split == "tune":
            out = out / "tune"
        cfg.update(weights_root="/tmp/hw2-variants-weights",
                   tokenizer=str(INPUTS / "tokenizer.json"),
                   rows=str(INPUTS / ("test.json" if split == "test" else "tune2048.json")),
                   output=str(out))
        if split == "tune":
            cfg["reference"] = str(INPUTS / ("dry-reference.json" if DRY else "tune2048-reference.json"))
        if DRY:
            cfg.update(device="cpu", dry_run=True)
        report = run_cold(cfg)
        reports[split] = report
        if split == "tune":
            quality = out / "quality.json"
            reports["quality"] = json.loads(quality.read_text()) if quality.exists() else None
        print(json.dumps(dict(variant=name, split=split, seconds=report["elapsed_seconds"],
            guards=report["guards"], peak_reserved_bytes=report.get("peak_reserved_bytes"),
            quality=reports.get("quality"))), flush=True)
    SUMMARY["variants"][name] = reports
    (ROOT / "variants/summary.json").write_text(json.dumps(SUMMARY,indent=2)+"\n")
print(json.dumps(dict(status="finished", variants=list(SUMMARY["variants"]))), flush=True)
